# ADTC 2026 — Gemma 3 270M Financial Intelligence Pipeline

**Goal:** turn Kenyan M-Pesa SMS logs into a small on-device financial intelligence system.

Pipeline:

**SMS → structured JSON → balance tracking → transaction DataFrame → cash-flow analytics → financial Q&A → LoRA fine-tuning → merged model → GGUF → llama.cpp**

### Important design decision

The LLM is responsible for **language understanding and structured extraction**.

Deterministic Python code is responsible for:
- balance reconciliation
- weekly/monthly totals
- spending by category/merchant
- cash-flow calculations
- loan affordability estimates
- validation of extracted financial fields

This separation makes financial arithmetic auditable instead of asking a tiny LLM to perform bookkeeping by itself.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/__results__.html
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/__notebook__.ipynb
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/__output__.json
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/custom.css
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/mpesa_extraction_v1/mpesa_extraction_v1.csv
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/mpesa_extraction_v1/mpesa_extraction_v1.json
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/mpesa_extraction_v1/dataset_quality_report.json
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/mpesa_extraction_v1/rejected_records.json
/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/mpesa_extraction_v1/mpesa_extraction_v1.jsonl
/kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2/config.json
/kaggle/input/models/google/gemma

In [2]:
# %% [code]
# =============================================================================
# CELL 1 — ENVIRONMENT / OPTIONAL DEPENDENCIES
# =============================================================================

!pip install -q -U "peft>=0.19.0" "datasets>=3.0.0" "transformers>=5.0.0" "scikit-learn" psutil

import os
import sys
import json
import math
import time
import random
import shutil
import gc
import re
import subprocess
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import psutil
import torch
import transformers
import datasets
import peft

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model

print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("PEFT         :", peft.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("CUDA version :", torch.version.cuda)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


ENVIRONMENT
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Transformers : 5.15.0
Datasets     : 5.0.1
PEFT         : 0.20.0
CUDA         : True
GPU          : Tesla T4
CUDA version : 12.8


In [3]:
# %% [code]
# =============================================================================
# CELL 2 — CONFIGURATION
# =============================================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_PATH = Path(
    "/kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/mpesa_extraction_v1/mpesa_extraction_v1.json"
)

BASE_MODEL = "google/gemma-3-270m-it"

# FP32 is retained for the T4 training/debugging stage.
MODEL_DTYPE = torch.float32

PROJECT_ROOT = Path("/kaggle/working")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
LORA_DIR = MODELS_DIR / "lora"
MERGED_DIR = MODELS_DIR / "merged"
GGUF_DIR = MODELS_DIR / "gguf"
REPORTS_DIR = PROJECT_ROOT / "reports"
EXPORT_DIR = PROJECT_ROOT / "export"

for d in [
    DATA_DIR, PROCESSED_DIR, MODELS_DIR, LORA_DIR, MERGED_DIR,
    GGUF_DIR, REPORTS_DIR, EXPORT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

SMS_COLUMN = None
TARGET_COLUMN = None

# LoRA
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training
LEARNING_RATE = 1e-4
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_LENGTH = 512
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Generation
MAX_NEW_TOKENS = 220

# Financial intelligence policy/configuration.
# This is an affordability heuristic, NOT a lender decision.
LOAN_SURPLUS_RATIO = 0.30
LOAN_TERM_MONTHS = 12
LOAN_SAFETY_BUFFER = 0.70

print("Base model :", BASE_MODEL)
print("Dataset    :", DATA_PATH)
print("Max length :", MAX_LENGTH)
print("Loan ratio :", LOAN_SURPLUS_RATIO)

Base model : google/gemma-3-270m-it
Dataset    : /kaggle/input/notebooks/wangapa106g/cleaning-the-gemma-sme-dataset/mpesa_extraction_v1/mpesa_extraction_v1.json
Max length : 512
Loan ratio : 0.3


In [4]:
# %% [code]
# =============================================================================
# CELL 3 — MEMORY UTILITIES
# =============================================================================

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_memory():
    if not torch.cuda.is_available():
        return {}
    return {
        "allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gb": round(torch.cuda.memory_reserved() / 2**30, 3),
    }

def print_memory(label=""):
    print(f"\nMEMORY — {label}")
    print(
        "CPU RSS:",
        round(psutil.Process(os.getpid()).memory_info().rss / 2**30, 3),
        "GB"
    )
    if torch.cuda.is_available():
        mem = gpu_memory()
        print("GPU allocated:", mem["allocated_gb"], "GB")
        print("GPU reserved :", mem["reserved_gb"], "GB")

In [5]:
# %% [code]
# =============================================================================
# CELL 4 — LOAD HUGGING FACE TOKEN FROM KAGGLE SECRETS
# =============================================================================

try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

    if not HF_TOKEN:
        raise ValueError("HF_TOKEN secret exists but is empty.")

    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✓ HF_TOKEN loaded from Kaggle Secrets")

except Exception as e:
    print("HF_TOKEN was not loaded.")
    print(type(e).__name__, ":", e)
    print("If the Gemma repository is already cached/publicly accessible, this may still work.")

✓ HF_TOKEN loaded from Kaggle Secrets


In [6]:
# %% [code]
# =============================================================================
# CELL 5 — LOAD DATASET
# =============================================================================

def load_dataset_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset does not exist: {path}")

    ext = path.suffix.lower()

    if ext == ".csv":
        return pd.read_csv(path)

    if ext == ".jsonl":
        return pd.read_json(path, lines=True)

    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

        if isinstance(obj, dict):
            for key, value in obj.items():
                if (
                    isinstance(value, list)
                    and value
                    and isinstance(value[0], dict)
                ):
                    print("Detected JSON container:", key)
                    return pd.DataFrame(value)
            return pd.DataFrame([obj])

    if ext == ".parquet":
        return pd.read_parquet(path)

    if ext == ".xlsx":
        return pd.read_excel(path)

    raise ValueError(f"Unsupported dataset format: {ext}")

df_raw = load_dataset_file(DATA_PATH)

print("=" * 80)
print("DATASET")
print("=" * 80)
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
display(df_raw.head(10))

DATASET
Shape: (200, 2)
Columns: ['input', 'target']


,input,target
0,"QRYEGGCMLP Confirmed. You have received Ksh10,...","{'entity': 'STANBIC BANK', 'amount': 10000.0, ..."
1,"Confirmed. You have received Ksh2,500.00 from ...","{'entity': 'Kennedy Waweru', 'amount': 2500.0,..."
2,"MPESA S6UYKNAYY8: You paid Ksh1,000.00 to STAN...","{'entity': 'STANDARD CHARTERED', 'amount': 100..."
3,STV9CJ6KN2 Confirmed. Ksh85.00 paid to KFC JUN...,"{'entity': 'KFC JUNCTION', 'amount': 85.0, 'ba..."
4,MPESA REF R8NQJKNEKS: Ian Wambui has sent you ...,"{'entity': 'Ian Wambui', 'amount': 150.0, 'bal..."
5,Ksh120.00 paid to GOODLIFE PHARMACY on 09/01/2...,"{'entity': 'GOODLIFE PHARMACY', 'amount': 120...."
6,Ksh750.00 paid to HOTPOINT APPLIANCES on 10/01...,"{'entity': 'HOTPOINT APPLIANCES', 'amount': 75..."
7,You have sent Ksh600.00 to FELIX MAINA on 11/0...,"{'entity': 'FELIX MAINA', 'amount': 600.0, 'ba..."
8,SAXAPZUG93 Confirmed. Ksh85.00 paid to HALTONS...,"{'entity': 'HALTONS PHARMACY', 'amount': 85.0,..."
9,You have sent Ksh400.00 to WINNIE MAINA on 13/...,"{'entity': 'WINNIE MAINA', 'amount': 400.0, 'b..."


In [7]:
# %% [code]
# =============================================================================
# CELL 6 — DETECT INPUT / TARGET COLUMNS
# =============================================================================

INPUT_HINTS = ["sms", "sms_text", "message", "text", "input", "prompt"]
OUTPUT_HINTS = ["output", "target", "label", "response", "json", "structured", "expected"]

def rank_columns(columns, hints):
    ranked = []
    for column in columns:
        name = str(column).lower()
        score = sum(hint in name for hint in hints)
        if score:
            ranked.append((score, column))
    ranked.sort(key=lambda x: (-x[0], str(x[1])))
    return [x[1] for x in ranked]

if SMS_COLUMN is None:
    candidates = rank_columns(df_raw.columns, INPUT_HINTS)
    if candidates:
        SMS_COLUMN = candidates[0]

if TARGET_COLUMN is None:
    candidates = rank_columns(df_raw.columns, OUTPUT_HINTS)
    if candidates:
        TARGET_COLUMN = candidates[0]

print("SMS column    :", SMS_COLUMN)
print("Target column :", TARGET_COLUMN)

if SMS_COLUMN is None or TARGET_COLUMN is None:
    raise ValueError(
        "Could not detect SMS/TARGET columns. Set SMS_COLUMN and TARGET_COLUMN manually."
    )

SMS column    : input
Target column : target


In [8]:
# %% [code]
# =============================================================================
# CELL 7 — JSON NORMALIZATION + FINANCIAL SCHEMA
# =============================================================================

def safe_json_loads(value):
    if isinstance(value, (dict, list)):
        return value
    if not isinstance(value, str):
        return None
    try:
        return json.loads(value)
    except Exception:
        return None

def canonical_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str,
    )

def normalize_target(value):
    parsed = safe_json_loads(value)
    return parsed if isinstance(parsed, (dict, list)) else None

# The model is explicitly taught to preserve balance_after whenever the SMS
# contains an M-Pesa "New M-PESA balance is ..." value.
#
# Recommended canonical object:
#
# {
#   transaction_id,
#   transaction_type,
#   amount,
#   entity,
#   entity_type,
#   category,
#   fee,
#   balance_before,
#   balance_after,
#   currency,
#   timestamp
# }
#
# balance_before may be null when it cannot be directly observed.
# balance_after is the observed account balance after the transaction.

def normalize_transaction_record(record):

    if not isinstance(record, dict):
        return {}

    return {
        "entity": record.get("entity"),
        "amount": record.get("amount"),
        "balance": record.get(
            "balance",
            record.get("balance_after")
        ),
        "date": record.get(
            "date",
            record.get("timestamp")
        ),
        "type": record.get(
            "type",
            record.get("transaction_type")
        ),
    }

In [9]:
# %% [code]
# =============================================================================
# CELL 8 — CLEAN DATA
# =============================================================================

clean = df_raw[[SMS_COLUMN, TARGET_COLUMN]].copy()

clean = clean.dropna(subset=[SMS_COLUMN, TARGET_COLUMN]).copy()

clean[SMS_COLUMN] = (
    clean[SMS_COLUMN]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

clean[TARGET_COLUMN] = clean[TARGET_COLUMN].map(normalize_target)

clean = clean[
    (clean[SMS_COLUMN].str.len() > 0)
    & clean[TARGET_COLUMN].map(lambda x: isinstance(x, dict))
].copy()

clean[TARGET_COLUMN] = clean[TARGET_COLUMN].map(normalize_transaction_record)

clean["_target_hash"] = clean[TARGET_COLUMN].map(canonical_json)

before = len(clean)
clean = clean.drop_duplicates(
    subset=[SMS_COLUMN, "_target_hash"],
    keep="first"
).reset_index(drop=True)

duplicates_removed = before - len(clean)
clean.drop(columns=["_target_hash"], inplace=True)

print("=" * 80)
print("CLEANING RESULT")
print("=" * 80)
print("Original rows      :", len(df_raw))
print("Valid rows         :", len(clean))
print("Duplicates removed :", duplicates_removed)

display(pd.json_normalize(clean[TARGET_COLUMN]).head())

CLEANING RESULT
Original rows      : 200
Valid rows         : 200
Duplicates removed : 0


,entity,amount,balance,date,type
0,STANBIC BANK,10000.0,140000.0,2026-06-01,income
1,Kennedy Waweru,2500.0,142500.0,2026-06-01,income
2,STANDARD CHARTERED,1000.0,141493.0,2026-06-01,expense
3,KFC JUNCTION,85.0,141408.0,2026-07-01,expense
4,Ian Wambui,150.0,141558.0,2026-08-01,income


In [10]:
# %% [code]
# =============================================================================
# CELL 9 — CONFLICT CHECK + LEAKAGE-SAFE SPLIT
# =============================================================================

conflict_check = clean.copy()
conflict_check["_target_hash"] = conflict_check[TARGET_COLUMN].map(canonical_json)

target_counts = (
    conflict_check
    .groupby(SMS_COLUMN)["_target_hash"]
    .nunique()
)

conflicting_sms = target_counts[target_counts > 1].index.tolist()

print("Conflicting SMS:", len(conflicting_sms))

if conflicting_sms:
    clean = clean[
        ~clean[SMS_COLUMN].isin(conflicting_sms)
    ].reset_index(drop=True)

clean["_sms_norm"] = (
    clean[SMS_COLUMN]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

unique_sms = clean["_sms_norm"].drop_duplicates().tolist()

train_sms, temp_sms = train_test_split(
    unique_sms,
    test_size=0.30,
    random_state=SEED,
)

val_sms, test_sms = train_test_split(
    temp_sms,
    test_size=0.50,
    random_state=SEED,
)

train_df = clean[clean["_sms_norm"].isin(train_sms)].copy()
val_df = clean[clean["_sms_norm"].isin(val_sms)].copy()
test_df = clean[clean["_sms_norm"].isin(test_sms)].copy()

for frame in [train_df, val_df, test_df]:
    frame.drop(columns=["_sms_norm"], inplace=True)
    frame.reset_index(drop=True, inplace=True)

print("=" * 80)
print("DATASET SPLIT")
print("=" * 80)
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

train_sms_set = set(train_df[SMS_COLUMN])
val_sms_set = set(val_df[SMS_COLUMN])
test_sms_set = set(test_df[SMS_COLUMN])

assert not train_sms_set & val_sms_set
assert not train_sms_set & test_sms_set
assert not val_sms_set & test_sms_set

print("✓ Leakage checks passed.")

Conflicting SMS: 0
DATASET SPLIT
Train      : 140
Validation : 30
Test       : 30
✓ Leakage checks passed.


In [11]:
# %% [code]
# =============================================================================
# CELL 10 — SAVE PROCESSED DATA
# =============================================================================

def write_jsonl(frame, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in frame.iterrows():
            record = {
                "input": row[SMS_COLUMN],
                "target": row[TARGET_COLUMN],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

write_jsonl(train_df, PROCESSED_DIR / "train.jsonl")
write_jsonl(val_df, PROCESSED_DIR / "validation.jsonl")
write_jsonl(test_df, PROCESSED_DIR / "test.jsonl")

print("Processed datasets saved to:", PROCESSED_DIR)

Processed datasets saved to: /kaggle/working/data/processed


## Financial intelligence layer

The next section is intentionally **not dependent on the LLM** for arithmetic.

Once each SMS has a structured record, the records become a transaction ledger. The ledger can then answer questions such as:

- Where is most of my money going?
- How much did I spend this week?
- What is my current/last observed balance?
- What are my total inflows and outflows?
- What is my net cash flow?
- Which merchants/categories consume the most money?
- What affordability amount does the configured loan heuristic produce?

This also means the system can continue working when the model is too small to reliably perform multi-step arithmetic.

## Training target improvement

The original notebook teaches extraction, but the generation test showed a weak/inconsistent schema.

The revised target explicitly teaches the model that **`balance` is a first-class financial field**.

We also add a strict output contract to every training example so that the model learns to return valid JSON rather than Python-like dictionaries.

In [12]:
# %% [code]
# =============================================================================
# CELL 18 — TOKENIZER
# =============================================================================

print("=" * 80)
print("LOADING TOKENIZER")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN if "HF_TOKEN" in globals() else None,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD:", tokenizer.pad_token)
print("EOS:", tokenizer.eos_token)

LOADING TOKENIZER


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Vocab size: 262144
PAD: <pad>
EOS: <eos>


In [13]:
# %% [code]
# =============================================================================
# CELL 19 — STRICT EXTRACTION PROMPT / TARGET FORMAT
# =============================================================================

SCHEMA_INSTRUCTION = """
Extract exactly five fields from the M-PESA SMS.

Return ONLY valid JSON.

Fields:
- entity: the exact person, business, merchant, bank, or organization involved
- amount: the transaction amount in KES
- balance: the M-PESA balance after the transaction
- date: the transaction date in YYYY-MM-DD format
- type: either "income" or "expense"

Rules:
- "received", "received from", "deposit", and money sent to the user = income
- "paid", "paid to", "sent", "bought", "withdrawn", and money sent by the user = expense
- entity must be the exact name appearing in the SMS
- do not invent an entity
- do not include transaction IDs
- do not include transaction fees
- do not include phone numbers
- do not include account numbers
- do not include merchant IDs
- do not include categories
- do not perform calculations
- do not add explanations
- do not add extra JSON fields

Required format:

{
  "entity": "...",
  "amount": 0,
  "balance": 0,
  "date": "YYYY-MM-DD",
  "type": "income"
}
""".strip()

def target_to_text(target):
    return json.dumps(
        normalize_transaction_record(target),
        ensure_ascii=False,
        separators=(",", ":"),
    )

def build_prompt(sms):
    messages = [{
        "role": "user",
        "content": f"{SCHEMA_INSTRUCTION}\n\nSMS:\n{sms}"
    }]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def build_example(row):
    return {
        "prompt": build_prompt(str(row[SMS_COLUMN])),
        "target": target_to_text(row[TARGET_COLUMN]),
    }

train_examples = [build_example(row) for _, row in train_df.iterrows()]
val_examples = [build_example(row) for _, row in val_df.iterrows()]
test_examples = [build_example(row) for _, row in test_df.iterrows()]

print(train_examples[0]["prompt"])
print("\nTARGET:")
print(train_examples[0]["target"])

<bos><start_of_turn>user
Extract exactly five fields from the M-PESA SMS.

Return ONLY valid JSON.

Fields:
- entity: the exact person, business, merchant, bank, or organization involved
- amount: the transaction amount in KES
- balance: the M-PESA balance after the transaction
- date: the transaction date in YYYY-MM-DD format
- type: either "income" or "expense"

Rules:
- "received", "received from", "deposit", and money sent to the user = income
- "paid", "paid to", "sent", "bought", "withdrawn", and money sent by the user = expense
- entity must be the exact name appearing in the SMS
- do not invent an entity
- do not include transaction IDs
- do not include transaction fees
- do not include phone numbers
- do not include account numbers
- do not include merchant IDs
- do not include categories
- do not perform calculations
- do not add explanations
- do not add extra JSON fields

Required format:

{
  "entity": "...",
  "amount": 0,
  "balance": 0,
  "date": "YYYY-MM-DD",
  "type":

In [14]:
# %% [code]
# =============================================================================
# CELL 20 — TOKENIZATION / COLLATOR
# =============================================================================

def tokenize_example(example):
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
    )["input_ids"]

    target_ids = tokenizer(
        example["target"],
        add_special_tokens=False,
    )["input_ids"]

    target_ids = target_ids + [tokenizer.eos_token_id]

    if len(target_ids) >= MAX_LENGTH:
        target_ids = target_ids[:MAX_LENGTH - 1] + [tokenizer.eos_token_id]

    max_prompt_length = MAX_LENGTH - len(target_ids)
    prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_dataset = Dataset.from_list([tokenize_example(x) for x in train_examples])
val_dataset = Dataset.from_list([tokenize_example(x) for x in val_examples])
test_dataset = Dataset.from_list([tokenize_example(x) for x in test_examples])

class CausalLMDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)

        input_ids = []
        attention_masks = []
        labels = []

        for feature in features:
            pad = max_len - len(feature["input_ids"])

            input_ids.append(
                feature["input_ids"] + [self.tokenizer.pad_token_id] * pad
            )
            attention_masks.append(
                feature["attention_mask"] + [0] * pad
            )
            labels.append(
                feature["labels"] + [-100] * pad
            )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

collator = CausalLMDataCollator(tokenizer)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

Train: 140
Val  : 30
Test : 30


In [15]:
# %% [code]
# =============================================================================
# CELL 21 — LOAD BASE MODEL + NUMERICAL CHECK
# =============================================================================

cleanup_memory()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    token=HF_TOKEN if "HF_TOKEN" in globals() else None,
)

if torch.cuda.is_available():
    base_model = base_model.cuda()

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

print("Model dtype :", next(base_model.parameters()).dtype)
print("Model device:", next(base_model.parameters()).device)
print_memory("BASE MODEL")

def check_model_parameters(model, name="model"):
    bad = []
    total = 0

    for param_name, param in model.named_parameters():
        total += param.numel()
        if not torch.isfinite(param).all():
            bad.append(param_name)

    print(f"{name}: checked {total:,} parameters; bad tensors={len(bad)}")

    if bad:
        print("First bad tensors:", bad[:20])
        raise RuntimeError(f"{name} contains NaN/Inf parameters.")

check_model_parameters(base_model, "BASE MODEL")

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Model dtype : torch.float32
Model device: cuda:0

MEMORY — BASE MODEL
CPU RSS: 1.779 GB
GPU allocated: 0.999 GB
GPU reserved : 1.0 GB
BASE MODEL: checked 268,098,176 parameters; bad tensors=0


In [16]:
# %% [code]
# =============================================================================
# CELL 22 — FORWARD TEST + LoRA
# =============================================================================
!pip install -q -U "torchao>=0.16.0"
def make_test_batch(dataset, batch_size=1):
    examples = [
        dataset[i]
        for i in range(min(batch_size, len(dataset)))
    ]
    batch = collator(examples)
    device = next(base_model.parameters()).device
    return {k: v.to(device) for k, v in batch.items()}

base_test_batch = make_test_batch(train_dataset)

base_model.eval()
with torch.no_grad():
    base_outputs = base_model(**base_test_batch)

assert torch.isfinite(base_outputs.logits).all()
assert torch.isfinite(base_outputs.loss)

print("Base forward loss:", base_outputs.loss.float().item())

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

check_model_parameters(model, "LoRA MODEL")

model.eval()
with torch.no_grad():
    lora_outputs = model(**base_test_batch)

assert torch.isfinite(lora_outputs.logits).all()
assert torch.isfinite(lora_outputs.loss)

print("LoRA forward loss:", lora_outputs.loss.float().item())

Base forward loss: 1.902050495147705
trainable params: 1,898,496 || all params: 269,996,672 || trainable%: 0.7032
LoRA MODEL: checked 269,996,672 parameters; bad tensors=0
LoRA forward loss: 1.902050495147705


In [17]:
# %% [code]
# =============================================================================
# CELL 23 — GRADIENT SANITY CHECK
# =============================================================================

model.train()
model.zero_grad(set_to_none=True)

outputs = model(**base_test_batch)
loss = outputs.loss

assert torch.isfinite(loss)

loss.backward()

bad_gradients = []
total_gradient_norm_sq = 0.0

for name, parameter in model.named_parameters():
    if not parameter.requires_grad or parameter.grad is None:
        continue

    if not torch.isfinite(parameter.grad).all():
        bad_gradients.append(name)

    g = parameter.grad.detach().float().norm(2).item()
    total_gradient_norm_sq += g ** 2

gradient_norm = math.sqrt(total_gradient_norm_sq)

print("Loss          :", loss.float().item())
print("Gradient norm :", gradient_norm)
print("Bad gradients :", len(bad_gradients))

if bad_gradients or not math.isfinite(gradient_norm):
    raise RuntimeError("Gradient sanity check failed.")

model.zero_grad(set_to_none=True)
print("✓ Gradient test passed.")

Loss          : 1.902050495147705
Gradient norm : 21.67793387438108
Bad gradients : 0
✓ Gradient test passed.


In [18]:
# %% [code]
# =============================================================================
# CELL 24 — TRAINING ARGUMENTS
# VERSION-ADAPTIVE / COMPATIBILITY SAFE
# =============================================================================

print("=" * 80)
print("BUILDING TRAINING ARGUMENTS")
print("=" * 80)

checkpoint_dir = MODELS_DIR / "checkpoints"

if checkpoint_dir.exists():
    shutil.rmtree(checkpoint_dir)

# -------------------------------------------------------------------------
# Inspect the installed TrainingArguments API
# -------------------------------------------------------------------------

import inspect

ta_params = inspect.signature(
    TrainingArguments.__init__
).parameters

print("Supported TrainingArguments parameters:")
print(", ".join(
    p for p in ta_params
    if p != "self"
))

# -------------------------------------------------------------------------
# Build arguments only when the installed Transformers version supports them
# -------------------------------------------------------------------------

training_kwargs = {
    "output_dir": str(checkpoint_dir),

    "num_train_epochs": NUM_EPOCHS,

    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,

    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,

    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,

    "max_grad_norm": MAX_GRAD_NORM,

    "fp16": False,
    "bf16": False,

    "logging_steps": 5,

    "save_total_limit": 2,

    "gradient_checkpointing": False,

    "report_to": "none",

    "remove_unused_columns": False,

    "seed": SEED,
    "data_seed": SEED,
}


# =========================================================================
# WARMUP
# =========================================================================

if "warmup_ratio" in ta_params:

    training_kwargs["warmup_ratio"] = WARMUP_RATIO

    print("✓ Using warmup_ratio")

elif "warmup_steps" in ta_params:

    # Calculate warmup steps manually.
    #
    # Number of optimizer steps per epoch:
    steps_per_epoch = math.ceil(
        len(train_dataset)
        / (
            TRAIN_BATCH_SIZE
            * GRADIENT_ACCUMULATION_STEPS
        )
    )

    total_steps = (
        steps_per_epoch
        * NUM_EPOCHS
    )

    warmup_steps = max(
        1,
        int(total_steps * WARMUP_RATIO)
    )

    training_kwargs["warmup_steps"] = warmup_steps

    print(
        f"✓ Using warmup_steps={warmup_steps} "
        f"(equivalent to ~{WARMUP_RATIO:.1%} warmup)"
    )

else:

    print("⚠ No warmup parameter supported.")


# =========================================================================
# EVALUATION STRATEGY
# =========================================================================

if "eval_strategy" in ta_params:

    training_kwargs["eval_strategy"] = "epoch"

    print("✓ Using eval_strategy")

elif "evaluation_strategy" in ta_params:

    training_kwargs["evaluation_strategy"] = "epoch"

    print("✓ Using evaluation_strategy")

else:

    print("⚠ Evaluation strategy unsupported.")


# =========================================================================
# SAVE STRATEGY
# =========================================================================

if "save_strategy" in ta_params:

    training_kwargs["save_strategy"] = "epoch"

    print("✓ Using save_strategy")

else:

    print("⚠ save_strategy unsupported.")


# =========================================================================
# BEST MODEL
# =========================================================================

if "load_best_model_at_end" in ta_params:

    training_kwargs["load_best_model_at_end"] = True

if "metric_for_best_model" in ta_params:

    training_kwargs["metric_for_best_model"] = "eval_loss"

if "greater_is_better" in ta_params:

    training_kwargs["greater_is_better"] = False


# =========================================================================
# LOGGING STRATEGY
# =========================================================================

if "logging_strategy" in ta_params:

    training_kwargs["logging_strategy"] = "steps"


# =========================================================================
# CREATE TRAINING ARGUMENTS
# =========================================================================

print("\nFinal TrainingArguments configuration:")
for key, value in training_kwargs.items():
    print(f"  {key:30s}: {value}")

training_args = TrainingArguments(
    **training_kwargs
)

print("\n✓ TrainingArguments created successfully.")


# =========================================================================
# CREATE TRAINER
# =========================================================================

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=collator,
)

print("✓ Trainer created successfully.")
print("=" * 80)

BUILDING TRAINING ARGUMENTS
Supported TrainingArguments parameters:
output_dir, per_device_train_batch_size, num_train_epochs, max_steps, learning_rate, lr_scheduler_type, lr_scheduler_kwargs, warmup_steps, optim, optim_args, weight_decay, adam_beta1, adam_beta2, adam_epsilon, optim_target_modules, gradient_accumulation_steps, average_tokens_across_devices, max_grad_norm, label_smoothing_factor, bf16, fp16, bf16_full_eval, fp16_full_eval, tf32, gradient_checkpointing, gradient_checkpointing_kwargs, torch_compile, torch_compile_backend, torch_compile_mode, use_liger_kernel, liger_kernel_config, use_cache, neftune_noise_alpha, torch_empty_cache_steps, auto_find_batch_size, logging_strategy, logging_steps, logging_first_step, log_on_each_node, logging_nan_inf_filter, include_num_input_tokens_seen, log_level, log_level_replica, disable_tqdm, report_to, run_name, project, trackio_space_id, trackio_bucket_id, trackio_static_space_id, eval_strategy, eval_steps, eval_delay, per_device_eval_bat

In [19]:
# %% [code]
# =============================================================================
# CELL 25 — FINAL PRE-TRAINING SANITY CHECK
# =============================================================================

trainer_model = trainer.model
first_parameter = next(trainer_model.parameters())

print("Model type :", type(trainer_model).__name__)
print("Device     :", first_parameter.device)
print("Dtype      :", first_parameter.dtype)

fresh_batch = collator([train_dataset[0]])
fresh_batch = {k: v.to(first_parameter.device) for k, v in fresh_batch.items()}

trainer_model.eval()
with torch.no_grad():
    outputs = trainer_model(**fresh_batch)

assert torch.isfinite(outputs.logits).all()
assert torch.isfinite(outputs.loss)

trainable = sum(p.numel() for p in trainer_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer_model.parameters())

print("Total params     :", f"{total:,}")
print("Trainable params :", f"{trainable:,}")
print("Trainable %      :", f"{100 * trainable / total:.4f}%")
print("Loss             :", outputs.loss.float().item())

trainer_model.train()
print("✓ ALL PRE-TRAINING CHECKS PASSED.")

Model type : PeftModelForCausalLM
Device     : cuda:0
Dtype      : torch.float32
Total params     : 269,996,672
Trainable params : 1,898,496
Trainable %      : 0.7032%
Loss             : 1.902050495147705
✓ ALL PRE-TRAINING CHECKS PASSED.


In [20]:
# %% [code]
# =============================================================================
# CELL 26 — TRAIN
# =============================================================================

print("=" * 80)
print("STARTING LoRA TRAINING")
print("=" * 80)

print_memory("BEFORE TRAINING")

train_result = trainer.train()

print("=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print("Training loss:", train_result.training_loss)

print_memory("AFTER TRAINING")

STARTING LoRA TRAINING

MEMORY — BEFORE TRAINING
CPU RSS: 2.15 GB
GPU allocated: 2.175 GB
GPU reserved : 4.48 GB


Epoch,Training Loss,Validation Loss
1,1.666162,0.373569
2,0.265191,0.099188
3,0.104493,0.069702
4,0.070435,0.056365
5,0.060221,0.051316


TRAINING COMPLETE
Training loss: 0.3403109073638916

MEMORY — AFTER TRAINING
CPU RSS: 2.805 GB
GPU allocated: 2.197 GB
GPU reserved : 9.922 GB


In [21]:
# %% [code]
# =============================================================================
# CELL 27 — EVALUATE + SAVE LoRA
# =============================================================================

evaluation = trainer.evaluate()

print("=" * 80)
print("VALIDATION")
print("=" * 80)
print(json.dumps(evaluation, indent=2, default=str))

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

print("LoRA adapter saved:", LORA_DIR)

Training Loss,Validation Loss,Epoch
0.060221,0.051316,5


VALIDATION
{
  "eval_loss": 0.05131591856479645
}
LoRA adapter saved: /kaggle/working/models/lora


In [22]:
# %% [code]
# =============================================================================
# CELL 28 — MERGE LoRA
# =============================================================================

print("=" * 80)
print("MERGING LoRA INTO BASE MODEL")
print("=" * 80)

model.eval()
merged_model = model.merge_and_unload()

check_model_parameters(merged_model, "MERGED MODEL")

merged_model.config.use_cache = True

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model:", MERGED_DIR)

MERGING LoRA INTO BASE MODEL
MERGED MODEL: checked 268,098,176 parameters; bad tensors=0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model: /kaggle/working/models/merged


## Post-training extraction test

A successful training loss does not automatically mean the model emits valid JSON.

The next cells:
1. generate an extraction,
2. parse it,
3. verify `balance_after`,
4. convert the generated records into the ledger,
5. run the same deterministic financial intelligence layer.

That gives a much more meaningful end-to-end test than checking loss alone.




In [23]:
# %% [code]
# =============================================================================
# CELL 29 — ROBUST JSON EXTRACTION FROM MODEL OUTPUT
# =============================================================================

def extract_first_json_object(text):
    text = text.strip()

    # Direct parse first.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Find the first balanced {...} region.
    start = text.find("{")
    if start < 0:
        return None

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i + 1]
                try:
                    obj = json.loads(candidate)
                    return obj if isinstance(obj, dict) else None
                except Exception:
                    return None

    return None

def generate_transaction_json(model, sms):
    messages = [{
        "role": "user",
        "content": f"{SCHEMA_INSTRUCTION}\n\nSMS:\n{sms}"
    }]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[
        0, inputs["input_ids"].shape[1]:
    ]

    text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    parsed = extract_first_json_object(text)

    return {
        "raw_text": text,
        "parsed": normalize_transaction_record(parsed or {}),
    }

In [24]:
# %% [code]
# =============================================================================
# CELL 30 — GENERATION TEST WITH 5-FIELD EXTRACTION
# =============================================================================

print("=" * 80)
print("GENERATION TEST — 5-FIELD M-PESA EXTRACTION")
print("=" * 80)

# -------------------------------------------------------------------------
# Select a real SMS from the training dataset
# -------------------------------------------------------------------------

sample_sms = str(
    train_df.iloc[8][SMS_COLUMN]
)


# -------------------------------------------------------------------------
# Generate structured transaction
# -------------------------------------------------------------------------

result = generate_transaction_json(
    merged_model,
    sample_sms,
)


parsed = result.get(
    "parsed"
)


# -------------------------------------------------------------------------
# Display SMS
# -------------------------------------------------------------------------

print("\nSMS:")
print("-" * 80)
print(sample_sms)


# -------------------------------------------------------------------------
# Display raw model output
# -------------------------------------------------------------------------

print("\nRAW MODEL OUTPUT:")
print("-" * 80)
print(result.get("raw_text"))


# -------------------------------------------------------------------------
# Display parsed JSON
# -------------------------------------------------------------------------

print("\nPARSED JSON:")
print("-" * 80)

if parsed is not None:

    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )

else:

    print("JSON parsing failed.")


# =============================================================================
# VALIDATE THE FIVE-FIELD CONTRACT
# =============================================================================

print("\n" + "=" * 80)
print("5-FIELD EXTRACTION VALIDATION")
print("=" * 80)


REQUIRED_FIELDS = {
    "entity",
    "amount",
    "balance",
    "date",
    "type",
}


if not isinstance(parsed, dict):

    print("✗ Parsed output is not a JSON object.")

else:

    actual_fields = set(
        parsed.keys()
    )

    missing_fields = (
        REQUIRED_FIELDS
        -
        actual_fields
    )

    extra_fields = (
        actual_fields
        -
        REQUIRED_FIELDS
    )


    print(
        "Required fields:",
        sorted(REQUIRED_FIELDS)
    )

    print(
        "Returned fields:",
        sorted(actual_fields)
    )


    if missing_fields:

        print(
            "✗ Missing fields:",
            sorted(missing_fields)
        )

    else:

        print(
            "✓ All five required fields present."
        )


    if extra_fields:

        print(
            "⚠ Extra fields:",
            sorted(extra_fields)
        )

    else:

        print(
            "✓ No extra fields."
        )


# =============================================================================
# DISPLAY EACH FINANCIAL FIELD
# =============================================================================

if isinstance(parsed, dict):

    print("\n" + "=" * 80)
    print("EXTRACTED FINANCIAL RECORD")
    print("=" * 80)

    print(
        "Entity  :",
        parsed.get("entity")
    )

    print(
        "Amount  :",
        parsed.get("amount")
    )

    print(
        "Balance :",
        parsed.get("balance")
    )

    print(
        "Date    :",
        parsed.get("date")
    )

    print(
        "Type    :",
        parsed.get("type")
    )


# =============================================================================
# TYPE VALIDATION
# =============================================================================

if isinstance(parsed, dict):

    transaction_type = str(
        parsed.get("type", "")
    ).lower().strip()


    if transaction_type in {
        "income",
        "expense",
    }:

        print(
            "\n✓ Transaction type is valid:",
            transaction_type
        )

    else:

        print(
            "\n⚠ Invalid transaction type:",
            repr(transaction_type)
        )


# =============================================================================
# BALANCE VALIDATION
# =============================================================================

if isinstance(parsed, dict):

    balance = parsed.get(
        "balance"
    )

    if balance is not None:

        print(
            "\n✓ Balance extracted:",
            balance
        )

    else:

        print(
            "\n⚠ WARNING: Model did not extract a balance."
        )


print("\n" + "=" * 80)
print("GENERATION TEST COMPLETE")
print("=" * 80)

GENERATION TEST — 5-FIELD M-PESA EXTRACTION

SMS:
--------------------------------------------------------------------------------
You have received Ksh20,000.00 from Ann Mueni. Transaction cost Ksh0.00. New balance Ksh159,583.00.

RAW MODEL OUTPUT:
--------------------------------------------------------------------------------
{"entity":"Ann Mueni","amount":20000.0,"balance":159583.0,"date":"2026-03-05","type":"income"}

PARSED JSON:
--------------------------------------------------------------------------------
{
  "entity": "Ann Mueni",
  "amount": 20000.0,
  "balance": 159583.0,
  "date": "2026-03-05",
  "type": "income"
}

5-FIELD EXTRACTION VALIDATION
Required fields: ['amount', 'balance', 'date', 'entity', 'type']
Returned fields: ['amount', 'balance', 'date', 'entity', 'type']
✓ All five required fields present.
✓ No extra fields.

EXTRACTED FINANCIAL RECORD
Entity  : Ann Mueni
Amount  : 20000.0
Balance : 159583.0
Date    : 2026-03-05
Type    : income

✓ Transaction type is v

In [25]:
# %% [code]
# =============================================================================
# CELL 31 — END-TO-END MODEL → 5-FIELD LEDGER TEST
# =============================================================================

print("=" * 80)
print("END-TO-END MODEL → LEDGER TEST")
print("=" * 80)


# =============================================================================
# 1. FIVE-FIELD RECORD NORMALIZATION
# =============================================================================

def model_records_to_dataframe(records):

    rows = []

    for record in records:

        if not isinstance(record, dict):
            continue

        # -------------------------------------------------------------
        # Extract ONLY the fields used by our financial intelligence
        # layer.
        # -------------------------------------------------------------

        entity = record.get("entity")

        amount = record.get("amount")

        balance = record.get("balance")

        date = record.get("date")

        transaction_type = record.get("type")


        # -------------------------------------------------------------
        # Normalize amount
        # -------------------------------------------------------------

        try:

            if amount is not None:
                amount = float(
                    str(amount)
                    .replace(",", "")
                    .replace("KES", "")
                    .strip()
                )

        except Exception:

            amount = np.nan


        # -------------------------------------------------------------
        # Normalize balance
        # -------------------------------------------------------------

        try:

            if balance is not None:
                balance = float(
                    str(balance)
                    .replace(",", "")
                    .replace("KES", "")
                    .strip()
                )

        except Exception:

            balance = np.nan


        # -------------------------------------------------------------
        # Normalize transaction type
        # -------------------------------------------------------------

        if transaction_type is None:

            transaction_type = "unknown"

        else:

            transaction_type = (
                str(transaction_type)
                .strip()
                .lower()
            )

            if transaction_type not in {
                "income",
                "expense"
            }:

                transaction_type = "unknown"


        # -------------------------------------------------------------
        # Normalize date
        # -------------------------------------------------------------

        date = pd.to_datetime(
            date,
            errors="coerce"
        )


        # -------------------------------------------------------------
        # Preserve exact entity text
        # -------------------------------------------------------------

        if entity is not None:

            entity = str(
                entity
            ).strip()

            if not entity:
                entity = None


        rows.append({

            "entity": entity,

            "amount": amount,

            "balance": balance,

            "date": date,

            "type": transaction_type,

        })


    # =========================================================================
    # BUILD DATAFRAME
    # =========================================================================

    ledger = pd.DataFrame(
        rows,
        columns=[
            "entity",
            "amount",
            "balance",
            "date",
            "type",
        ],
    )


    if not ledger.empty:

        ledger["amount"] = pd.to_numeric(
            ledger["amount"],
            errors="coerce"
        )

        ledger["balance"] = pd.to_numeric(
            ledger["balance"],
            errors="coerce"
        )

        ledger["date"] = pd.to_datetime(
            ledger["date"],
            errors="coerce"
        )

        ledger["type"] = (
            ledger["type"]
            .astype("string")
        )

        ledger["entity"] = (
            ledger["entity"]
            .astype("string")
        )

        ledger = ledger.sort_values(
            "date",
            na_position="last"
        ).reset_index(
            drop=True
        )


    return ledger


# =============================================================================
# 2. RUN MODEL ON TEST SMS
# =============================================================================

sample_results = []

test_sample = test_df[
    SMS_COLUMN
].head(
    min(10, len(test_df))
)


print(
    f"Testing {len(test_sample)} SMS messages..."
)

print()


for i, sms in enumerate(
    test_sample,
    start=1
):

    sms = str(sms)

    print(
        f"[{i}/{len(test_sample)}] Generating..."
    )


    generated_record = generate_transaction_json(
        merged_model,
        sms,
    )


    parsed = generated_record.get(
        "parsed"
    )


    # -------------------------------------------------------------
    # Only accept dictionary JSON objects
    # -------------------------------------------------------------

    if not isinstance(
        parsed,
        dict
    ):

        print(
            "  ✗ Invalid JSON"
        )

        continue


    # -------------------------------------------------------------
    # Keep only our five target fields.
    #
    # This prevents accidental fields such as:
    # transaction_id
    # fee
    # phone_number
    # category
    # etc.
    # from entering the financial ledger.
    # -------------------------------------------------------------

    clean_record = {

        "entity": parsed.get(
            "entity"
        ),

        "amount": parsed.get(
            "amount"
        ),

        "balance": parsed.get(
            "balance"
        ),

        "date": parsed.get(
            "date"
        ),

        "type": parsed.get(
            "type"
        ),

    }


    sample_results.append(
        clean_record
    )


# =============================================================================
# 3. CONVERT MODEL OUTPUT → DATAFRAME
# =============================================================================

model_ledger = model_records_to_dataframe(
    sample_results
)


print()
print("=" * 80)
print("MODEL-GENERATED LEDGER")
print("=" * 80)

display(
    model_ledger
)


# =============================================================================
# 4. BASIC EXTRACTION QUALITY CHECK
# =============================================================================

print()
print("=" * 80)
print("EXTRACTION QUALITY")
print("=" * 80)


if model_ledger.empty:

    print(
        "✗ No valid model records were produced."
    )

else:

    total = len(
        model_ledger
    )

    entity_ok = (
        model_ledger["entity"]
        .notna()
        .sum()
    )

    amount_ok = (
        model_ledger["amount"]
        .notna()
        .sum()
    )

    balance_ok = (
        model_ledger["balance"]
        .notna()
        .sum()
    )

    date_ok = (
        model_ledger["date"]
        .notna()
        .sum()
    )

    type_ok = (
        model_ledger["type"]
        .isin([
            "income",
            "expense"
        ])
        .sum()
    )


    print(
        f"Records generated : {total}"
    )

    print(
        f"Entity extracted  : {entity_ok}/{total}"
    )

    print(
        f"Amount extracted  : {amount_ok}/{total}"
    )

    print(
        f"Balance extracted : {balance_ok}/{total}"
    )

    print(
        f"Date extracted    : {date_ok}/{total}"
    )

    print(
        f"Type classified   : {type_ok}/{total}"
    )


# =============================================================================
# 5. FINANCIAL ANALYSIS DIRECTLY ON FIVE-FIELD LEDGER
# =============================================================================

print()
print("=" * 80)
print("MODEL FINANCIAL ANALYSIS")
print("=" * 80)


if model_ledger.empty:

    print(
        "No records available for financial analysis."
    )

else:

    # -------------------------------------------------------------------------
    # Income
    # -------------------------------------------------------------------------

    total_income = model_ledger.loc[
        model_ledger["type"] == "income",
        "amount"
    ].sum()


    # -------------------------------------------------------------------------
    # Expenses
    # -------------------------------------------------------------------------

    total_expenses = model_ledger.loc[
        model_ledger["type"] == "expense",
        "amount"
    ].sum()


    # -------------------------------------------------------------------------
    # Net cash flow
    # -------------------------------------------------------------------------

    net_cash_flow = (
        total_income
        -
        total_expenses
    )


    # -------------------------------------------------------------------------
    # Latest observed balance
    # -------------------------------------------------------------------------

    valid_balances = (
        model_ledger["balance"]
        .dropna()
    )


    if len(valid_balances):

        latest_balance = float(
            valid_balances.iloc[-1]
        )

    else:

        latest_balance = None


    # -------------------------------------------------------------------------
    # Spending by entity
    # -------------------------------------------------------------------------

    expenses = model_ledger[
        model_ledger["type"] == "expense"
    ].copy()


    if not expenses.empty:

        spending_by_entity = (
            expenses
            .groupby(
                "entity",
                dropna=False
            )["amount"]
            .sum()
            .sort_values(
                ascending=False
            )
        )

    else:

        spending_by_entity = pd.Series(
            dtype=float
        )


    # -------------------------------------------------------------------------
    # Print summary
    # -------------------------------------------------------------------------

    print(
        f"Transactions   : {len(model_ledger)}"
    )

    print(
        f"Income         : KES {total_income:,.2f}"
    )

    print(
        f"Expenses       : KES {total_expenses:,.2f}"
    )

    print(
        f"Net cash flow  : KES {net_cash_flow:,.2f}"
    )


    if latest_balance is not None:

        print(
            f"Latest balance : KES {latest_balance:,.2f}"
        )

    else:

        print(
            "Latest balance : N/A"
        )


    print()
    print(
        "TOP SPENDING ENTITIES"
    )
    print(
        "-" * 80
    )


    if spending_by_entity.empty:

        print(
            "No expense transactions available."
        )

    else:

        display(
            spending_by_entity
            .head(10)
            .rename(
                "total_spent"
            )
            .to_frame()
        )


# =============================================================================
# 6. FINAL RESULT
# =============================================================================

print()
print("=" * 80)
print("✓ END-TO-END MODEL TEST COMPLETE")
print("=" * 80)

print(
    "Target schema:"
)

print(
    [
        "entity",
        "amount",
        "balance",
        "date",
        "type",
    ]
)

print()
print(
    "The financial analysis uses ONLY the five extracted fields."
)

END-TO-END MODEL → LEDGER TEST
Testing 10 SMS messages...

[1/10] Generating...
[2/10] Generating...
[3/10] Generating...
[4/10] Generating...
[5/10] Generating...
[6/10] Generating...
[7/10] Generating...
[8/10] Generating...
[9/10] Generating...
[10/10] Generating...

MODEL-GENERATED LEDGER


,entity,amount,balance,date,type
0,TOTALENERGIES NAIROBI,2300.0,127693.0,2026-01-28,expense
1,Emmanuel Rono,2000.0,91042.0,2026-02-27,expense
2,SHELL WESTLANDS,5600.0,122006.0,2026-03-02,expense
3,IAN KIPROP,3000.0,171417.0,2026-03-05,expense
4,HUSTLER FUND,120.0,67707.0,2026-03-05,expense
5,Cynthia Chebet,300.0,128008.0,2026-03-13,income
6,NCBA BANK,15000.0,68182.0,2026-05-03,income
7,HUSTLER FUND,350.0,67832.0,2026-06-03,expense
8,GOODLIFE PHARMACY,120.0,141438.0,2026-09-01,expense
9,WINNIE MAINA,400.0,139583.0,NaT,expense



EXTRACTION QUALITY
Records generated : 10
Entity extracted  : 10/10
Amount extracted  : 10/10
Balance extracted : 10/10
Date extracted    : 9/10
Type classified   : 10/10

MODEL FINANCIAL ANALYSIS
Transactions   : 10
Income         : KES 15,300.00
Expenses       : KES 13,890.00
Net cash flow  : KES 1,410.00
Latest balance : KES 139,583.00

TOP SPENDING ENTITIES
--------------------------------------------------------------------------------


,total_spent
entity,
SHELL WESTLANDS,5600.0
IAN KIPROP,3000.0
TOTALENERGIES NAIROBI,2300.0
Emmanuel Rono,2000.0
HUSTLER FUND,470.0
WINNIE MAINA,400.0
GOODLIFE PHARMACY,120.0



✓ END-TO-END MODEL TEST COMPLETE
Target schema:
['entity', 'amount', 'balance', 'date', 'type']

The financial analysis uses ONLY the five extracted fields.


## Natural-language financial questions

The safest architecture is:

**User question → intent extraction → deterministic analytics → small LLM verbalization**

For example, the question *“How much did I spend this week?”* should be answered from the DataFrame's calculated total, not from the model guessing from raw SMS text.

The functions below implement the analytical side of that interface.

In [26]:
# %% [code]
# =============================================================================
# CELL 32 — NATURAL-LANGUAGE FINANCIAL QUESTION ENGINE
# =============================================================================
#
# Architecture:
#
# User question
#       ↓
# Intent detection
#       ↓
# Deterministic Python analytics
#       ↓
# Financial answer
#
# The LLM is NOT used for arithmetic.
#
# Current ledger schema:
#
#     entity
#     amount
#     balance
#     date
#     type
#
# where type ∈ {"income", "expense"}
# =============================================================================


def _financial_summary(ledger):

    if ledger is None or ledger.empty:

        return {
            "transactions": 0,
            "income": 0.0,
            "expenses": 0.0,
            "net_cash_flow": 0.0,
            "latest_balance": None,
        }


    df = ledger.copy()


    # -------------------------------------------------------------------------
    # Ensure numeric columns
    # -------------------------------------------------------------------------

    df["amount"] = pd.to_numeric(
        df["amount"],
        errors="coerce"
    )

    df["balance"] = pd.to_numeric(
        df["balance"],
        errors="coerce"
    )


    # -------------------------------------------------------------------------
    # Income
    # -------------------------------------------------------------------------

    income = df.loc[
        df["type"].astype(str).str.lower() == "income",
        "amount"
    ].sum()


    # -------------------------------------------------------------------------
    # Expenses
    # -------------------------------------------------------------------------

    expenses = df.loc[
        df["type"].astype(str).str.lower() == "expense",
        "amount"
    ].sum()


    # -------------------------------------------------------------------------
    # Latest observed balance
    # -------------------------------------------------------------------------

    valid_balances = (
        df["balance"]
        .dropna()
    )


    latest_balance = (
        float(valid_balances.iloc[-1])
        if len(valid_balances)
        else None
    )


    return {
        "transactions": int(len(df)),
        "income": round(float(income), 2),
        "expenses": round(float(expenses), 2),
        "net_cash_flow": round(
            float(income - expenses),
            2
        ),
        "latest_balance": latest_balance,
    }


# =============================================================================
# WEEKLY / PERIOD SPENDING
# =============================================================================

def _period_expenses(ledger, days):

    if ledger is None or ledger.empty:
        return 0.0


    df = ledger.copy()


    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )


    df["amount"] = pd.to_numeric(
        df["amount"],
        errors="coerce"
    )


    valid_dates = df["date"].dropna()


    if valid_dates.empty:
        return 0.0


    # Use the latest transaction date as the reference point.
    # This is important for historical/test SMS datasets.
    latest_date = valid_dates.max()


    cutoff = (
        latest_date
        -
        pd.Timedelta(days=days)
    )


    period_df = df[
        (df["date"] >= cutoff)
        &
        (
            df["type"]
            .astype(str)
            .str.lower()
            == "expense"
        )
    ]


    return round(
        float(
            period_df["amount"]
            .sum()
        ),
        2
    )


# =============================================================================
# TOP SPENDING ENTITIES
# =============================================================================

def _top_spending_entities(
    ledger,
    top_n=5
):

    if ledger is None or ledger.empty:

        return pd.DataFrame(
            columns=[
                "entity",
                "amount",
                "share",
            ]
        )


    df = ledger.copy()


    df["amount"] = pd.to_numeric(
        df["amount"],
        errors="coerce"
    )


    expenses = df[
        df["type"]
        .astype(str)
        .str.lower()
        .eq("expense")
    ].copy()


    if expenses.empty:

        return pd.DataFrame(
            columns=[
                "entity",
                "amount",
                "share",
            ]
        )


    # -------------------------------------------------------------------------
    # Missing entities are kept separate rather than inventing a name.
    # -------------------------------------------------------------------------

    expenses["entity"] = (
        expenses["entity"]
        .fillna("Unknown entity")
        .astype(str)
        .str.strip()
    )


    grouped = (
        expenses
        .groupby(
            "entity",
            as_index=False
        )["amount"]
        .sum()
        .sort_values(
            "amount",
            ascending=False
        )
        .reset_index(drop=True)
    )


    total = grouped["amount"].sum()


    if total > 0:

        grouped["share"] = (
            grouped["amount"]
            /
            total
            *
            100
        )

    else:

        grouped["share"] = 0.0


    grouped["amount"] = grouped[
        "amount"
    ].round(2)

    grouped["share"] = grouped[
        "share"
    ].round(1)


    return grouped.head(top_n)


# =============================================================================
# LOAN AFFORDABILITY
# =============================================================================

def _loan_heuristic(ledger):

    summary = _financial_summary(
        ledger
    )


    income = summary["income"]
    expenses = summary["expenses"]
    balance = summary["latest_balance"]


    # -------------------------------------------------------------------------
    # No financial activity
    # -------------------------------------------------------------------------

    if summary["transactions"] == 0:

        return {
            "status": "insufficient_data",
            "requested_amount": None,
            "illustrative_loan_amount": 0.0,
        }


    # -------------------------------------------------------------------------
    # Surplus-based heuristic.
    #
    # IMPORTANT:
    # This is only an analytical heuristic, NOT a lender decision.
    #
    # We deliberately keep the arithmetic in Python.
    # -------------------------------------------------------------------------

    surplus = income - expenses


    if surplus <= 0:

        affordable = 0.0

    else:

        # Configured 30% surplus heuristic
        affordable = surplus * 0.30


    return {

        "status": "calculated",

        "income": round(
            income,
            2
        ),

        "expenses": round(
            expenses,
            2
        ),

        "surplus": round(
            surplus,
            2
        ),

        "latest_balance": balance,

        "illustrative_loan_amount": round(
            affordable,
            2
        ),

    }


# =============================================================================
# MAIN QUESTION ENGINE
# =============================================================================

def answer_financial_question(
    question,
    ledger
):

    if ledger is None:

        return (
            "No transaction ledger is available."
        )


    q = (
        str(question)
        .lower()
        .strip()
    )


    # =========================================================================
    # 1. SPENDING BY ENTITY
    # =========================================================================

    if any(
        phrase in q
        for phrase in [
            "where is most",
            "where am i spending",
            "spending most",
            "most of my money",
            "biggest expense",
            "largest expense",
            "where do i spend",
            "who do i spend",
        ]
    ):

        breakdown = _top_spending_entities(
            ledger,
            top_n=5
        )


        if breakdown.empty:

            return (
                "There is not enough expense data "
                "to identify where most of your money is going."
            )


        top = breakdown.iloc[0]


        return (
            f"Your largest spending entity is "
            f"{top['entity']}, where you spent "
            f"KES {top['amount']:,.2f}. "
            f"That represents approximately "
            f"{top['share']:.1f}% of your recorded expenses."
        )


    # =========================================================================
    # 2. WEEKLY SPENDING
    # =========================================================================

    if any(
        phrase in q
        for phrase in [
            "how much did i spend this week",
            "spent this week",
            "spending this week",
            "weekly spending",
        ]
    ):

        amount = _period_expenses(
            ledger,
            days=7
        )


        return (
            f"You spent approximately "
            f"KES {amount:,.2f} "
            f"in the last 7 days of available transaction data."
        )


    # =========================================================================
    # 3. MONTHLY SPENDING
    # =========================================================================

    if any(
        phrase in q
        for phrase in [
            "how much did i spend this month",
            "spent this month",
            "monthly spending",
        ]
    ):

        amount = _period_expenses(
            ledger,
            days=30
        )


        return (
            f"You spent approximately "
            f"KES {amount:,.2f} "
            f"in the last 30 days of available transaction data."
        )


    # =========================================================================
    # 4. BALANCE
    # =========================================================================

    if any(
        phrase in q
        for phrase in [
            "balance",
            "how much do i have",
            "how much money do i have",
            "account balance",
            "mpesa balance",
        ]
    ):

        summary = _financial_summary(
            ledger
        )


        balance = summary[
            "latest_balance"
        ]


        if balance is None:

            return (
                "No valid M-PESA balance "
                "was extracted from the available SMS records."
            )


        return (
            f"Your last observed M-PESA balance "
            f"is KES {balance:,.2f}."
        )


    # =========================================================================
    # 5. CASH FLOW
    # =========================================================================

    if any(
        phrase in q
        for phrase in [
            "cash flow",
            "cashflow",
            "inflow",
            "outflow",
            "income",
            "financial position",
        ]
    ):

        summary = _financial_summary(
            ledger
        )


        return (
            f"Your recorded income is "
            f"KES {summary['income']:,.2f}, "
            f"your recorded expenses are "
            f"KES {summary['expenses']:,.2f}, "
            f"and your net cash flow is "
            f"KES {summary['net_cash_flow']:,.2f}."
        )


    # =========================================================================
    # 6. LOAN AFFORDABILITY
    # =========================================================================

    if any(
        phrase in q
        for phrase in [
            "loan",
            "borrow",
            "credit",
            "how much can i apply",
            "can i afford",
            "loan amount",
        ]
    ):

        loan = _loan_heuristic(
            ledger
        )


        if loan["status"] == "insufficient_data":

            return (
                "There is not enough transaction data "
                "to calculate the configured affordability heuristic."
            )


        amount = loan[
            "illustrative_loan_amount"
        ]


        return (
            f"Based on the configured Python affordability "
            f"heuristic, the illustrative amount is "
            f"KES {amount:,.2f}. "
            f"This is an analytical estimate, not a lender "
            f"approval or guaranteed borrowing amount."
        )


    # =========================================================================
    # 7. FALLBACK
    # =========================================================================

    return (
        "I can answer questions about your balance, "
        "weekly or monthly spending, income, expenses, "
        "cash flow, spending by entity, and the configured "
        "loan affordability heuristic."
    )


# =============================================================================
# TEST AGAINST THE ACTUAL MODEL LEDGER
# =============================================================================

# IMPORTANT:
# Cell 31 creates `model_ledger`.
# We deliberately use that here instead of the old `ledger` variable.

question_ledger = model_ledger


example_questions = [

    "Where is most of my money going?",

    "How much did I spend this week?",

    "How much did I spend this month?",

    "What is my account balance?",

    "Show me my cash flow",

    "What amount of loan can I apply?",

]


print("=" * 80)
print("NATURAL-LANGUAGE FINANCIAL QUESTIONS")
print("=" * 80)


for question in example_questions:

    print()
    print("Q:", question)

    answer = answer_financial_question(
        question,
        question_ledger
    )

    print("A:", answer)


print()
print("=" * 80)
print("✓ FINANCIAL QUESTION ENGINE READY")
print("=" * 80)

NATURAL-LANGUAGE FINANCIAL QUESTIONS

Q: Where is most of my money going?
A: Your largest spending entity is SHELL WESTLANDS, where you spent KES 5,600.00. That represents approximately 40.3% of your recorded expenses.

Q: How much did I spend this week?
A: You spent approximately KES 120.00 in the last 7 days of available transaction data.

Q: How much did I spend this month?
A: You spent approximately KES 120.00 in the last 30 days of available transaction data.

Q: What is my account balance?
A: Your last observed M-PESA balance is KES 139,583.00.

Q: Show me my cash flow
A: Your recorded income is KES 15,300.00, your recorded expenses are KES 13,890.00, and your net cash flow is KES 1,410.00.

Q: What amount of loan can I apply?
A: Based on the configured Python affordability heuristic, the illustrative amount is KES 423.00. This is an analytical estimate, not a lender approval or guaranteed borrowing amount.

✓ FINANCIAL QUESTION ENGINE READY


In [27]:
# %% [code]
# =============================================================================
# CELL 33 — FINAL FINANCIAL INTELLIGENCE ARTIFACTS
# =============================================================================
#
# CANONICAL MODEL OUTPUT:
#
#   entity
#   amount
#   balance
#   date
#   type
#
# type:
#   income
#   expense
#
# IMPORTANT:
# - No balance_before
# - No balance_after
# - No transaction_id
# - No transaction fees
# - No categories
# - No LLM arithmetic
#
# Python/pandas performs ALL financial calculations.
# =============================================================================

print("=" * 80)
print("FINAL FINANCIAL INTELLIGENCE ARTIFACTS")
print("=" * 80)


# =============================================================================
# 1. FIND THE ACTUAL MODEL LEDGER
# =============================================================================
#
# Notebook execution order can leave one of these variables undefined.
#
# Priority:
#
#   model_ledger
#       ↓
#   ledger
#       ↓
#   sample_results
#
# We then create ONE canonical variable:
#
#       ledger
#
# Every later financial-analysis cell should use `ledger`.
# =============================================================================

source_name = None

# -------------------------------------------------------------------------
# OPTION A — model_ledger already exists
# -------------------------------------------------------------------------

if (
    "model_ledger" in globals()
    and isinstance(model_ledger, pd.DataFrame)
):

    ledger = model_ledger.copy()
    source_name = "model_ledger"


# -------------------------------------------------------------------------
# OPTION B — ledger already exists
# -------------------------------------------------------------------------

elif (
    "ledger" in globals()
    and isinstance(ledger, pd.DataFrame)
):

    ledger = ledger.copy()
    source_name = "ledger"


# -------------------------------------------------------------------------
# OPTION C — sample_results exists
# -------------------------------------------------------------------------

elif (
    "sample_results" in globals()
    and isinstance(sample_results, list)
    and len(sample_results) > 0
):

    if "records_to_dataframe" not in globals():

        raise RuntimeError(
            "sample_results exists, but records_to_dataframe() "
            "is not defined. Run the 5-field extraction parser cell first."
        )

    ledger = records_to_dataframe(
        sample_results
    )

    source_name = "sample_results"


# -------------------------------------------------------------------------
# NOTHING AVAILABLE
# -------------------------------------------------------------------------

else:

    raise RuntimeError(
        "\nNo transaction data is currently available.\n\n"

        "Run the model generation test first.\n\n"

        "Expected one of:\n"
        "  1. model_ledger\n"
        "  2. ledger\n"
        "  3. sample_results\n\n"

        "For example, run your model → ledger test cell first."
    )


print()
print("Ledger source :", source_name)
print("Raw rows      :", len(ledger))
print("Raw columns   :", list(ledger.columns))


# =============================================================================
# 2. SUPPORT OLD COLUMN NAMES ONLY WHEN NECESSARY
# =============================================================================
#
# The CURRENT target remains:
#
#   entity
#   amount
#   balance
#   date
#   type
#
# These mappings are only defensive compatibility.
# They are NOT part of the model target.
# =============================================================================

if (
    "balance" not in ledger.columns
    and "balance_after" in ledger.columns
):

    ledger["balance"] = ledger[
        "balance_after"
    ]

    print(
        "Compatibility mapping: "
        "balance_after → balance"
    )


if (
    "date" not in ledger.columns
    and "timestamp" in ledger.columns
):

    ledger["date"] = ledger[
        "timestamp"
    ]

    print(
        "Compatibility mapping: "
        "timestamp → date"
    )


if (
    "type" not in ledger.columns
    and "transaction_type" in ledger.columns
):

    ledger["type"] = ledger[
        "transaction_type"
    ]

    print(
        "Compatibility mapping: "
        "transaction_type → type"
    )


# =============================================================================
# 3. REQUIRE THE FIVE-FIELD SCHEMA
# =============================================================================

REQUIRED_COLUMNS = [
    "entity",
    "amount",
    "balance",
    "date",
    "type",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in ledger.columns
]

if missing_columns:

    raise ValueError(
        "\nThe current model ledger does not contain the required "
        "5-field extraction schema.\n\n"

        f"Missing : {missing_columns}\n"
        f"Found   : {list(ledger.columns)}\n\n"

        "Expected:\n"
        "  entity\n"
        "  amount\n"
        "  balance\n"
        "  date\n"
        "  type"
    )


# =============================================================================
# 4. CREATE THE CANONICAL FIVE-FIELD LEDGER
# =============================================================================

ledger = ledger[
    REQUIRED_COLUMNS
].copy()


# =============================================================================
# 5. NORMALIZE DATA TYPES
# =============================================================================

ledger["entity"] = (
    ledger["entity"]
    .astype("string")
    .str.strip()
)

ledger["amount"] = pd.to_numeric(
    ledger["amount"],
    errors="coerce",
)

ledger["balance"] = pd.to_numeric(
    ledger["balance"],
    errors="coerce",
)

ledger["date"] = pd.to_datetime(
    ledger["date"],
    errors="coerce",
)

ledger["type"] = (
    ledger["type"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# =============================================================================
# 6. NORMALIZE TRANSACTION TYPE
# =============================================================================
#
# The model should ideally already output:
#
#   income
#   expense
#
# These mappings simply make the analysis robust to occasional
# model variations.
# =============================================================================

INCOME_TERMS = [
    "income",
    "received",
    "receive",
    "deposit",
    "credit",
    "inflow",
    "cash_in",
    "transfer_in",
]

EXPENSE_TERMS = [
    "expense",
    "paid",
    "payment",
    "pay",
    "sent",
    "send",
    "bought",
    "buy",
    "withdraw",
    "withdrawal",
    "debit",
    "outflow",
    "cash_out",
    "transfer_out",
]


def normalize_type(value):

    if pd.isna(value):
        return "unknown"

    text = str(value).strip().lower()

    # Exact / phrase matching
    if any(
        term in text
        for term in INCOME_TERMS
    ):
        return "income"

    if any(
        term in text
        for term in EXPENSE_TERMS
    ):
        return "expense"

    return "unknown"


ledger["type"] = ledger[
    "type"
].map(
    normalize_type
)


# =============================================================================
# 7. SORT TRANSACTIONS
# =============================================================================

ledger = (
    ledger
    .sort_values(
        "date",
        na_position="last"
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 8. CREATE ANALYSIS LEDGER
# =============================================================================
#
# We retain the original five-field ledger for extraction evaluation.
#
# For financial arithmetic, amount must exist.
# =============================================================================

analysis_ledger = ledger[
    ledger["amount"].notna()
].copy()

analysis_ledger = (
    analysis_ledger
    .reset_index(drop=True)
)


# =============================================================================
# 9. EXTRACTION QUALITY REPORT
# =============================================================================

total_records = len(ledger)

quality = {

    "total_records": total_records,

    "valid_entity": int(
        ledger["entity"].notna().sum()
    ),

    "valid_amount": int(
        ledger["amount"].notna().sum()
    ),

    "valid_balance": int(
        ledger["balance"].notna().sum()
    ),

    "valid_date": int(
        ledger["date"].notna().sum()
    ),

    "valid_type": int(
        ledger["type"]
        .isin(["income", "expense"])
        .sum()
    ),

    "income_records": int(
        (
            ledger["type"]
            == "income"
        ).sum()
    ),

    "expense_records": int(
        (
            ledger["type"]
            == "expense"
        ).sum()
    ),

    "unknown_type_records": int(
        (
            ledger["type"]
            == "unknown"
        ).sum()
    ),
}


# =============================================================================
# 10. BALANCE ANALYTICS
# =============================================================================

def calculate_balance_analytics_compact(df):

    balances = (
        pd.to_numeric(
            df["balance"],
            errors="coerce"
        )
        .dropna()
    )

    if balances.empty:

        return {
            "first_observed_balance": None,
            "last_observed_balance": None,
            "balance_change": None,
            "balance_observation_count": 0,
        }

    first_balance = float(
        balances.iloc[0]
    )

    last_balance = float(
        balances.iloc[-1]
    )

    return {

        "first_observed_balance": round(
            first_balance,
            2
        ),

        "last_observed_balance": round(
            last_balance,
            2
        ),

        "balance_change": round(
            last_balance - first_balance,
            2
        ),

        "balance_observation_count": int(
            len(balances)
        ),
    }


balance_summary = (
    calculate_balance_analytics_compact(
        analysis_ledger
    )
)


# =============================================================================
# 11. CASH-FLOW ANALYTICS
# =============================================================================

def calculate_cash_flow(
    df,
    days=None
):

    working = df.copy()

    if working.empty:

        return {
            "period": (
                f"last_{days}_days"
                if days is not None
                else "all_available_data"
            ),

            "transactions": 0,

            "income": 0.0,

            "expenses": 0.0,

            "net_cash_flow": 0.0,
        }


    # ---------------------------------------------------------------------
    # Optional date filtering
    # ---------------------------------------------------------------------

    if (
        days is not None
        and working["date"].notna().any()
    ):

        latest_date = (
            working["date"].max()
        )

        cutoff = (
            latest_date
            -
            pd.Timedelta(days=days)
        )

        working = working[
            working["date"] >= cutoff
        ].copy()


    # ---------------------------------------------------------------------
    # Income
    # ---------------------------------------------------------------------

    income = working.loc[
        working["type"] == "income",
        "amount"
    ].sum()


    # ---------------------------------------------------------------------
    # Expenses
    # ---------------------------------------------------------------------

    expenses = working.loc[
        working["type"] == "expense",
        "amount"
    ].sum()


    return {

        "period": (
            f"last_{days}_days"
            if days is not None
            else "all_available_data"
        ),

        "transactions": int(
            len(working)
        ),

        "income": round(
            float(income),
            2
        ),

        "expenses": round(
            float(expenses),
            2
        ),

        "net_cash_flow": round(
            float(
                income - expenses
            ),
            2
        ),
    }


all_cash_flow = (
    calculate_cash_flow(
        analysis_ledger
    )
)

weekly_cash_flow = (
    calculate_cash_flow(
        analysis_ledger,
        days=7
    )
)

monthly_cash_flow = (
    calculate_cash_flow(
        analysis_ledger,
        days=30
    )
)


# =============================================================================
# 12. SPENDING BY ENTITY
# =============================================================================
#
# This is intentionally ENTITY-based.
#
# Example:
#
#   Safaricom
#   Naivas
#   KPLC
#   ZUKU
#   John Kamau
#
# The model does NOT need to understand categories.
# =============================================================================

def spending_by_entity(
    df,
    top_n=10
):

    expenses = df[
        df["type"] == "expense"
    ].copy()

    expenses = expenses[
        expenses["amount"].notna()
    ]

    expenses = expenses[
        expenses["entity"].notna()
    ]

    expenses = expenses[
        expenses["entity"].str.len() > 0
    ]

    if expenses.empty:

        return pd.DataFrame(
            columns=[
                "entity",
                "amount",
                "share_percent",
            ]
        )


    grouped = (
        expenses
        .groupby(
            "entity"
        )["amount"]
        .sum()
        .sort_values(
            ascending=False
        )
        .reset_index()
    )


    total_spending = (
        grouped["amount"].sum()
    )


    if total_spending > 0:

        grouped[
            "share_percent"
        ] = (
            grouped["amount"]
            /
            total_spending
            *
            100
        )

    else:

        grouped[
            "share_percent"
        ] = 0.0


    grouped["amount"] = (
        grouped["amount"]
        .round(2)
    )

    grouped[
        "share_percent"
    ] = (
        grouped[
            "share_percent"
        ]
        .round(2)
    )


    return grouped.head(
        top_n
    )


entity_spending = (
    spending_by_entity(
        analysis_ledger,
        top_n=10
    )
)


# =============================================================================
# 13. SAVINGS RATE
# =============================================================================

income = (
    all_cash_flow["income"]
)

expenses = (
    all_cash_flow["expenses"]
)

if income > 0:

    savings_rate = round(
        (
            income - expenses
        )
        /
        income
        *
        100,
        2
    )

else:

    savings_rate = None


# =============================================================================
# 14. TOP SPENDING ENTITY
# =============================================================================

if not entity_spending.empty:

    top_entity = str(
        entity_spending.iloc[0][
            "entity"
        ]
    )

    top_entity_amount = float(
        entity_spending.iloc[0][
            "amount"
        ]
    )

else:

    top_entity = None
    top_entity_amount = None


# =============================================================================
# 15. FINANCIAL SNAPSHOT
# =============================================================================

financial_snapshot = {

    "schema": {
        "entity": (
            "Exact entity involved in the transaction."
        ),

        "amount": (
            "Transaction amount in KES."
        ),

        "balance": (
            "Observed account/M-PESA balance."
        ),

        "date": (
            "Transaction date."
        ),

        "type": (
            "income or expense."
        ),
    },

    "extraction_quality": quality,

    "cash_flow": all_cash_flow,

    "last_7_days": weekly_cash_flow,

    "last_30_days": monthly_cash_flow,

    "balance": balance_summary,

    "savings_rate_percent": savings_rate,

    "top_spending_entity": top_entity,

    "top_spending_entity_amount": (
        top_entity_amount
    ),

    "top_spending_entities": (
        entity_spending
        .to_dict(
            orient="records"
        )
    ),
}


# =============================================================================
# 16. EXPORT DIRECTORY
# =============================================================================

EXPORT_DIR = Path(
    EXPORT_DIR
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 17. SAVE EXACT FIVE-FIELD LEDGER
# =============================================================================

ledger_path = (
    EXPORT_DIR
    /
    "transaction_ledger.csv"
)

ledger.to_csv(
    ledger_path,
    index=False
)


# =============================================================================
# 18. SAVE ANALYSIS LEDGER
# =============================================================================

analysis_ledger_path = (
    EXPORT_DIR
    /
    "analysis_ledger.csv"
)

analysis_ledger.to_csv(
    analysis_ledger_path,
    index=False
)


# =============================================================================
# 19. SAVE FINANCIAL SNAPSHOT
# =============================================================================

snapshot_path = (
    EXPORT_DIR
    /
    "financial_intelligence_snapshot.json"
)

with open(
    snapshot_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        financial_snapshot,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# =============================================================================
# 20. SAVE SPENDING ANALYSIS
# =============================================================================

spending_path = (
    EXPORT_DIR
    /
    "spending_by_entity.csv"
)

entity_spending.to_csv(
    spending_path,
    index=False
)


# =============================================================================
# 21. IMPORTANT — MAKE `ledger` THE CANONICAL NOTEBOOK VARIABLE
# =============================================================================

globals()["ledger"] = ledger

globals()["analysis_ledger"] = (
    analysis_ledger
)


# =============================================================================
# 22. DISPLAY FINAL LEDGER
# =============================================================================

print()
print("=" * 80)
print("FINAL FIVE-FIELD LEDGER")
print("=" * 80)

display(
    ledger
)


# =============================================================================
# 23. FINANCIAL SUMMARY
# =============================================================================

print()
print("=" * 80)
print("FINANCIAL INTELLIGENCE SUMMARY")
print("=" * 80)

print(
    f"Transactions        : {len(analysis_ledger)}"
)

print(
    f"Income              : "
    f"KES {income:,.2f}"
)

print(
    f"Expenses            : "
    f"KES {expenses:,.2f}"
)

print(
    f"Net cash flow       : "
    f"KES {all_cash_flow['net_cash_flow']:,.2f}"
)

if (
    balance_summary[
        "last_observed_balance"
    ]
    is not None
):

    print(
        f"Latest balance      : "
        f"KES "
        f"{balance_summary['last_observed_balance']:,.2f}"
    )

else:

    print(
        "Latest balance      : N/A"
    )


if top_entity:

    print(
        f"Top spending entity : "
        f"{top_entity} "
        f"(KES {top_entity_amount:,.2f})"
    )

else:

    print(
        "Top spending entity : N/A"
    )


if savings_rate is not None:

    print(
        f"Savings rate        : "
        f"{savings_rate:.2f}%"
    )

else:

    print(
        "Savings rate        : N/A"
    )


# =============================================================================
# 24. TOP SPENDING ENTITIES
# =============================================================================

print()
print("=" * 80)
print("TOP SPENDING ENTITIES")
print("=" * 80)

if entity_spending.empty:

    print(
        "No valid expense transactions available."
    )

else:

    display(
        entity_spending
    )


# =============================================================================
# 25. EXTRACTION QUALITY
# =============================================================================

print()
print("=" * 80)
print("EXTRACTION QUALITY")
print("=" * 80)

for key, value in quality.items():

    print(
        f"{key:25s}: {value}"
    )


# =============================================================================
# 26. FILES
# =============================================================================

print()
print("=" * 80)
print("ARTIFACTS SAVED")
print("=" * 80)

print(
    "Ledger          :",
    ledger_path
)

print(
    "Analysis ledger :",
    analysis_ledger_path
)

print(
    "Snapshot        :",
    snapshot_path
)

print(
    "Spending        :",
    spending_path
)

print()
print(
    "✓ `ledger` is now the canonical 5-field financial ledger."
)

print(
    "✓ `analysis_ledger` contains records usable for financial arithmetic."
)

print(
    "✓ No balance_after/balance_before assumptions are used."
)

FINAL FINANCIAL INTELLIGENCE ARTIFACTS

Ledger source : model_ledger
Raw rows      : 10
Raw columns   : ['entity', 'amount', 'balance', 'date', 'type']

FINAL FIVE-FIELD LEDGER


,entity,amount,balance,date,type
0,TOTALENERGIES NAIROBI,2300.0,127693.0,2026-01-28,expense
1,Emmanuel Rono,2000.0,91042.0,2026-02-27,expense
2,SHELL WESTLANDS,5600.0,122006.0,2026-03-02,expense
3,IAN KIPROP,3000.0,171417.0,2026-03-05,expense
4,HUSTLER FUND,120.0,67707.0,2026-03-05,expense
5,Cynthia Chebet,300.0,128008.0,2026-03-13,income
6,NCBA BANK,15000.0,68182.0,2026-05-03,income
7,HUSTLER FUND,350.0,67832.0,2026-06-03,expense
8,GOODLIFE PHARMACY,120.0,141438.0,2026-09-01,expense
9,WINNIE MAINA,400.0,139583.0,NaT,expense



FINANCIAL INTELLIGENCE SUMMARY
Transactions        : 10
Income              : KES 15,300.00
Expenses            : KES 13,890.00
Net cash flow       : KES 1,410.00
Latest balance      : KES 139,583.00
Top spending entity : SHELL WESTLANDS (KES 5,600.00)
Savings rate        : 9.22%

TOP SPENDING ENTITIES


,entity,amount,share_percent
0,SHELL WESTLANDS,5600.0,40.32
1,IAN KIPROP,3000.0,21.60
2,TOTALENERGIES NAIROBI,2300.0,16.56
3,Emmanuel Rono,2000.0,14.40
4,HUSTLER FUND,470.0,3.38
5,WINNIE MAINA,400.0,2.88
6,GOODLIFE PHARMACY,120.0,0.86



EXTRACTION QUALITY
total_records            : 10
valid_entity             : 10
valid_amount             : 10
valid_balance            : 10
valid_date               : 9
valid_type               : 10
income_records           : 2
expense_records          : 8
unknown_type_records     : 0

ARTIFACTS SAVED
Ledger          : /kaggle/working/export/transaction_ledger.csv
Analysis ledger : /kaggle/working/export/analysis_ledger.csv
Snapshot        : /kaggle/working/export/financial_intelligence_snapshot.json
Spending        : /kaggle/working/export/spending_by_entity.csv

✓ `ledger` is now the canonical 5-field financial ledger.
✓ `analysis_ledger` contains records usable for financial arithmetic.
✓ No balance_after/balance_before assumptions are used.


# GGUF / llama.cpp export

GGUF export is a separate deployment stage.

The intended flow is:

**merged Hugging Face model → FP16 GGUF → Q4_K_M GGUF**

`llama.cpp` provides `convert_hf_to_gguf.py` for the first step and `llama-quantize` for the second. Q4_K_M is a practical small-model deployment target for an 8 GB-class laptop.

The notebook does **not** assume a preinstalled llama.cpp binary. The next cell can clone and build it when network access is available.

For a production submission, keep the unquantized merged model as a reproducibility artifact and use the quantized GGUF for on-device inference.

In [28]:
# %% [code]
# =============================================================================
# CELL 34 — PREPARE LLAMA.CPP
# =============================================================================

LLAMA_CPP_DIR = PROJECT_ROOT / "llama.cpp"

if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp...")
    subprocess.run(
        [
            "git", "clone",
            "--depth", "1",
            "https://github.com/ggml-org/llama.cpp.git",
            str(LLAMA_CPP_DIR),
        ],
        check=True,
    )
else:
    print("llama.cpp already exists:", LLAMA_CPP_DIR)

print("llama.cpp directory ready.")

Cloning llama.cpp...


Cloning into '/kaggle/working/llama.cpp'...


llama.cpp directory ready.


In [29]:
# %% [code]
# =============================================================================
# CELL 35 — INSTALL CONVERTER DEPENDENCIES
# =============================================================================

requirements = LLAMA_CPP_DIR / "requirements" / "requirements-convert_hf_to_gguf.txt"

if requirements.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )
    print("✓ Converter requirements installed.")
else:
    print("Requirements file not found at:", requirements)
    print("Inspect the current llama.cpp checkout before continuing.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you hav

✓ Converter requirements installed.


In [30]:
# %% [code]
# =============================================================================
# CELL 36A — LOCATE GEMMA TOKENIZER
# =============================================================================

from pathlib import Path

print("=" * 80)
print("SEARCHING KAGGLE FILESYSTEM FOR GEMMA TOKENIZER")
print("=" * 80)

search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

matches = []

for root in search_roots:

    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for pattern in [
        "tokenizer.model",
        "tokenizer.json",
        "tokenizer_config.json",
    ]:

        found = list(
            root.rglob(pattern)
        )

        for path in found:

            matches.append(path)

            print(
                f"{pattern:25s} -> {path}"
            )


print("\n" + "=" * 80)
print("TOKENIZER.MODEL RESULTS")
print("=" * 80)

spm_files = [
    p for p in matches
    if p.name == "tokenizer.model"
]

if spm_files:

    print(
        f"✓ Found {len(spm_files)} tokenizer.model file(s)"
    )

    for p in spm_files:
        print(" ", p)

else:

    print(
        "❌ No tokenizer.model found anywhere under /kaggle/input or /kaggle/working"
    )


print("\n" + "=" * 80)
print("TOKENIZER.JSON RESULTS")
print("=" * 80)

json_files = [
    p for p in matches
    if p.name == "tokenizer.json"
]

for p in json_files:
    print(" ", p)

SEARCHING KAGGLE FILESYSTEM FOR GEMMA TOKENIZER

Searching: /kaggle/input
tokenizer.model           -> /kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2/tokenizer.model
tokenizer.json            -> /kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2/tokenizer.json
tokenizer_config.json     -> /kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2/tokenizer_config.json

Searching: /kaggle/working
tokenizer.json            -> /kaggle/working/models/lora/tokenizer.json
tokenizer.json            -> /kaggle/working/models/merged/tokenizer.json
tokenizer.json            -> /kaggle/working/models/checkpoints/checkpoint-36/tokenizer.json
tokenizer.json            -> /kaggle/working/models/checkpoints/checkpoint-45/tokenizer.json
tokenizer_config.json     -> /kaggle/working/models/lora/tokenizer_config.json
tokenizer_config.json     -> /kaggle/working/models/merged/tokenizer_config.json
tokenizer_config.json     -> /kaggle/working/models/checkpoints/checkpoint

In [31]:
# %% [code]
# =============================================================================
# CELL 36 — GEMMA 3 → FP16 GGUF
# FIXED: USE ORIGINAL KAGGLE SENTENCEPIECE TOKENIZER
# =============================================================================

import shutil
import subprocess
import sys
import json
from pathlib import Path


print("=" * 80)
print("GEMMA 3 → FP16 GGUF CONVERSION")
print("=" * 80)


# =============================================================================
# 1. PATHS
# =============================================================================

MERGED_DIR = Path(MERGED_DIR)
GGUF_DIR = Path(GGUF_DIR)
LLAMA_CPP_DIR = Path(LLAMA_CPP_DIR)

# Original Gemma 3 tokenizer discovered in Kaggle
TOKENIZER_SOURCE = Path(
    "/kaggle/input/models/google/gemma-3/"
    "transformers/gemma-3-270m/2"
)

TOKENIZER_MODEL = (
    TOKENIZER_SOURCE /
    "tokenizer.model"
)

CONVERTER = (
    LLAMA_CPP_DIR /
    "convert_hf_to_gguf.py"
)

CONVERSION_DIR = (
    GGUF_DIR /
    "hf_gemma3_conversion"
)

HF_GGUF = (
    GGUF_DIR /
    "gemma3-financial-intelligence-f16.gguf"
)


# =============================================================================
# 2. VALIDATE PATHS
# =============================================================================

print("\nChecking paths...")

checks = {
    "Merged model": MERGED_DIR,
    "GGUF directory": GGUF_DIR,
    "llama.cpp converter": CONVERTER,
    "Tokenizer directory": TOKENIZER_SOURCE,
    "tokenizer.model": TOKENIZER_MODEL,
}


for name, path in checks.items():

    exists = path.exists()

    print(
        f"{name:25s}: "
        f"{'✓' if exists else '✗'} "
        f"{path}"
    )

    if not exists:

        raise FileNotFoundError(
            f"{name} does not exist:\n{path}"
        )


GGUF_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INSPECT MERGED MODEL
# =============================================================================

print("\n" + "=" * 80)
print("MERGED MODEL")
print("=" * 80)

config_path = (
    MERGED_DIR /
    "config.json"
)

if not config_path.exists():

    raise FileNotFoundError(
        "Merged model is missing config.json"
    )


with open(
    config_path,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)


print(
    "Architecture:",
    config.get("architectures")
)

print(
    "Model type:",
    config.get("model_type")
)

print(
    "Vocab size:",
    config.get("vocab_size")
)

print(
    "Hidden size:",
    config.get("hidden_size")
)

print(
    "Layers:",
    config.get("num_hidden_layers")
)


# =============================================================================
# 4. CREATE CLEAN CONVERSION DIRECTORY
# =============================================================================

print("\n" + "=" * 80)
print("CREATING CLEAN CONVERSION DIRECTORY")
print("=" * 80)


if CONVERSION_DIR.exists():

    shutil.rmtree(
        CONVERSION_DIR
    )


CONVERSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 5. COPY MODEL FILES
# =============================================================================

print("\nCopying trained model...")


# Copy every model/config file from merged directory
for source in MERGED_DIR.iterdir():

    if not source.is_file():
        continue

    # We deliberately replace tokenizer files below
    # with the original Gemma tokenizer.
    if source.name in {
        "tokenizer.json",
        "tokenizer_config.json",
        "tokenizer.model",
        "special_tokens_map.json",
        "added_tokens.json",
    }:
        continue

    destination = (
        CONVERSION_DIR /
        source.name
    )

    shutil.copy2(
        source,
        destination
    )

    print(
        f"✓ {source.name}"
    )


# =============================================================================
# 6. COPY ORIGINAL GEMMA TOKENIZER
# =============================================================================

print("\n" + "=" * 80)
print("INSTALLING ORIGINAL GEMMA TOKENIZER")
print("=" * 80)


# tokenizer.model
shutil.copy2(
    TOKENIZER_SOURCE / "tokenizer.model",
    CONVERSION_DIR / "tokenizer.model"
)

print(
    "✓ tokenizer.model"
)


# tokenizer.json
if (
    TOKENIZER_SOURCE /
    "tokenizer.json"
).exists():

    shutil.copy2(
        TOKENIZER_SOURCE / "tokenizer.json",
        CONVERSION_DIR / "tokenizer.json"
    )

    print(
        "✓ tokenizer.json"
    )


# tokenizer_config.json
if (
    TOKENIZER_SOURCE /
    "tokenizer_config.json"
).exists():

    shutil.copy2(
        TOKENIZER_SOURCE /
        "tokenizer_config.json",

        CONVERSION_DIR /
        "tokenizer_config.json"
    )

    print(
        "✓ tokenizer_config.json"
    )


# special tokens
if (
    TOKENIZER_SOURCE /
    "special_tokens_map.json"
).exists():

    shutil.copy2(
        TOKENIZER_SOURCE /
        "special_tokens_map.json",

        CONVERSION_DIR /
        "special_tokens_map.json"
    )

    print(
        "✓ special_tokens_map.json"
    )


# chat template
chat_template = (
    MERGED_DIR /
    "chat_template.jinja"
)

if chat_template.exists():

    shutil.copy2(
        chat_template,
        CONVERSION_DIR /
        "chat_template.jinja"
    )

    print(
        "✓ chat_template.jinja"
    )


# =============================================================================
# 7. VERIFY CONVERSION DIRECTORY
# =============================================================================

print("\n" + "=" * 80)
print("CONVERSION DIRECTORY")
print("=" * 80)


for file in sorted(
    CONVERSION_DIR.iterdir()
):

    if file.is_file():

        size_mb = (
            file.stat().st_size /
            (1024 ** 2)
        )

        print(
            f"{file.name:40s}"
            f"{size_mb:10.2f} MB"
        )


# =============================================================================
# 8. CRITICAL TOKENIZER CHECK
# =============================================================================

print("\n" + "=" * 80)
print("TOKENIZER CHECK")
print("=" * 80)


conversion_tokenizer = (
    CONVERSION_DIR /
    "tokenizer.model"
)


if not conversion_tokenizer.exists():

    raise RuntimeError(
        "CRITICAL: tokenizer.model was not copied."
    )


tokenizer_size = (
    conversion_tokenizer.stat().st_size
)


print(
    "✓ tokenizer.model present"
)

print(
    f"Tokenizer size: "
    f"{tokenizer_size / (1024 ** 2):.2f} MB"
)


# =============================================================================
# 9. REMOVE PREVIOUS GGUF
# =============================================================================

if HF_GGUF.exists():

    print(
        "\nRemoving previous GGUF..."
    )

    HF_GGUF.unlink()


# =============================================================================
# 10. BUILD CONVERSION COMMAND
# =============================================================================

cmd = [
    sys.executable,
    str(CONVERTER),
    str(CONVERSION_DIR),
    "--outfile",
    str(HF_GGUF),
    "--outtype",
    "f16",
]


print("\n" + "=" * 80)
print("CONVERSION COMMAND")
print("=" * 80)

print(
    " ".join(
        map(str, cmd)
    )
)


# =============================================================================
# 11. RUN CONVERSION
# =============================================================================

print("\n" + "=" * 80)
print("RUNNING LLAMA.CPP")
print("=" * 80)


result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


# Print everything
print(
    result.stdout
)


# =============================================================================
# 12. CHECK CONVERSION
# =============================================================================

if result.returncode != 0:

    print("\n" + "=" * 80)
    print("❌ CONVERSION FAILED")
    print("=" * 80)

    raise RuntimeError(
        f"""
llama.cpp GGUF conversion failed.

Exit code:
{result.returncode}

The complete converter output is printed above.
"""
    )


# =============================================================================
# 13. VERIFY OUTPUT
# =============================================================================

if not HF_GGUF.exists():

    raise RuntimeError(
        """
llama.cpp exited successfully but the
GGUF file was not created.
"""
    )


gguf_size_gb = (
    HF_GGUF.stat().st_size /
    (1024 ** 3)
)


print("\n" + "=" * 80)
print("✓ FP16 GGUF CREATED")
print("=" * 80)

print(
    "Output:"
)

print(
    HF_GGUF
)

print(
    f"\nSize: {gguf_size_gb:.3f} GB"
)

GEMMA 3 → FP16 GGUF CONVERSION

Checking paths...
Merged model             : ✓ /kaggle/working/models/merged
GGUF directory           : ✓ /kaggle/working/models/gguf
llama.cpp converter      : ✓ /kaggle/working/llama.cpp/convert_hf_to_gguf.py
Tokenizer directory      : ✓ /kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2
tokenizer.model          : ✓ /kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2/tokenizer.model

MERGED MODEL
Architecture: ['Gemma3ForCausalLM']
Model type: gemma3_text
Vocab size: 262144
Hidden size: 640
Layers: 18

CREATING CLEAN CONVERSION DIRECTORY

Copying trained model...
✓ model.safetensors
✓ config.json
✓ generation_config.json
✓ chat_template.jinja

INSTALLING ORIGINAL GEMMA TOKENIZER
✓ tokenizer.model
✓ tokenizer.json
✓ tokenizer_config.json
✓ special_tokens_map.json
✓ chat_template.jinja

CONVERSION DIRECTORY
chat_template.jinja                           0.00 MB
config.json                                   0.00 MB
generation_confi

In [32]:
# %% [code]
# =============================================================================
# CELL 37 — BUILD LLAMA-QUANTIZE
# ROBUST KAGGLE VERSION
# =============================================================================

import subprocess
import shutil
from pathlib import Path


print("=" * 80)
print("BUILDING / LOCATING LLAMA-QUANTIZE")
print("=" * 80)


# =============================================================================
# 1. LLAMA.CPP PATHS
# =============================================================================

LLAMA_CPP_DIR = Path(LLAMA_CPP_DIR).resolve()

BUILD_DIR = (
    LLAMA_CPP_DIR /
    "build"
)

BIN_DIR = (
    BUILD_DIR /
    "bin"
)


print("llama.cpp source:")
print(LLAMA_CPP_DIR)

print("\nBuild directory:")
print(BUILD_DIR)

print("\nBinary directory:")
print(BIN_DIR)


# =============================================================================
# 2. VALIDATE LLAMA.CPP SOURCE
# =============================================================================

print("\n" + "=" * 80)
print("VALIDATING LLAMA.CPP SOURCE")
print("=" * 80)


if not LLAMA_CPP_DIR.exists():

    raise FileNotFoundError(
        f"""
llama.cpp directory does not exist:

{LLAMA_CPP_DIR}
"""
    )


CMAKE_FILE = (
    LLAMA_CPP_DIR /
    "CMakeLists.txt"
)


if not CMAKE_FILE.exists():

    raise FileNotFoundError(
        f"""
CMakeLists.txt was not found in:

{LLAMA_CPP_DIR}

Expected:

{CMAKE_FILE}
"""
    )


print(
    "✓ CMakeLists.txt found:"
)

print(
    CMAKE_FILE
)


# =============================================================================
# 3. SEARCH FOR EXISTING QUANTIZER
# =============================================================================

print("\n" + "=" * 80)
print("SEARCHING FOR EXISTING LLAMA-QUANTIZE")
print("=" * 80)


quantizer_candidates = [

    # Linux
    BUILD_DIR /
    "bin" /
    "llama-quantize",

    # Older llama.cpp layout
    BUILD_DIR /
    "bin" /
    "quantize",

    # Alternative build layout
    BUILD_DIR /
    "llama-quantize",

    # System PATH
    Path(
        shutil.which("llama-quantize")
        or "/__nonexistent__"
    ),

]


quantizer = None


for candidate in quantizer_candidates:

    if candidate.exists():

        quantizer = candidate.resolve()

        print(
            "✓ Found:",
            quantizer
        )

        break


# =============================================================================
# 4. CONFIGURE CMAKE
# =============================================================================

if quantizer is None:

    print("\nNo existing llama-quantize found.")

    print("\n" + "=" * 80)
    print("CONFIGURING CMAKE")
    print("=" * 80)


    BUILD_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


    configure_cmd = [

        "cmake",

        # Explicit source directory
        "-S",
        str(LLAMA_CPP_DIR),

        # Explicit build directory
        "-B",
        str(BUILD_DIR),

        # Kaggle-friendly build
        "-DGGML_NATIVE=OFF",

        # CPU build is sufficient for quantization
        "-DGGML_CUDA=OFF",

        # Build only what we need
        "-DLLAMA_BUILD_TOOLS=ON",

    ]


    print(
        "Running:"
    )

    print(
        " ".join(
            map(str, configure_cmd)
        )
    )


    configure_result = subprocess.run(
        configure_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )


    print(
        "\n" +
        configure_result.stdout
    )


    if configure_result.returncode != 0:

        raise RuntimeError(
            f"""
CMake configuration failed.

Exit code:
{configure_result.returncode}

The complete CMake output is printed above.
"""
        )


# =============================================================================
# 5. BUILD LLAMA-QUANTIZE
# =============================================================================

if quantizer is None:

    print("\n" + "=" * 80)
    print("BUILDING LLAMA-QUANTIZE")
    print("=" * 80)


    build_cmd = [

        "cmake",

        "--build",
        str(BUILD_DIR),

        "--config",
        "Release",

        "--target",
        "llama-quantize",

        "--",

        "-j2",

    ]


    print(
        "Running:"
    )

    print(
        " ".join(
            map(str, build_cmd)
        )
    )


    build_result = subprocess.run(
        build_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )


    print(
        "\n" +
        build_result.stdout
    )


    if build_result.returncode != 0:

        raise RuntimeError(
            f"""
llama-quantize build failed.

Exit code:
{build_result.returncode}

The complete build output is printed above.
"""
        )


# =============================================================================
# 6. FIND QUANTIZER AFTER BUILD
# =============================================================================

print("\n" + "=" * 80)
print("LOCATING QUANTIZER AFTER BUILD")
print("=" * 80)


quantizer_candidates = [

    BUILD_DIR /
    "bin" /
    "llama-quantize",

    BUILD_DIR /
    "bin" /
    "quantize",

    BUILD_DIR /
    "llama-quantize",

]


quantizer = None


for candidate in quantizer_candidates:

    if candidate.exists():

        quantizer = candidate.resolve()

        break


if quantizer is None:

    # Last-resort recursive search
    found = list(
        BUILD_DIR.rglob(
            "llama-quantize"
        )
    )

    if found:

        quantizer = found[0].resolve()


if quantizer is None:

    raise FileNotFoundError(
        f"""
llama-quantize was not found after building.

Searched:

{BUILD_DIR}
"""
    )


# =============================================================================
# 7. MAKE EXECUTABLE
# =============================================================================

try:

    quantizer.chmod(
        quantizer.stat().st_mode | 0o111
    )

except Exception as e:

    print(
        "Warning: could not change executable permissions:",
        e
    )


# =============================================================================
# 8. TEST QUANTIZER
# =============================================================================

print("\n" + "=" * 80)
print("TESTING LLAMA-QUANTIZE")
print("=" * 80)


test_result = subprocess.run(
    [
        str(quantizer),
        "--help",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


print(
    test_result.stdout[:5000]
)


if test_result.returncode not in [0, 1]:

    raise RuntimeError(
        f"""
llama-quantize executable could not be started.

Exit code:
{test_result.returncode}
"""
    )


# =============================================================================
# 9. FINAL RESULT
# =============================================================================

print("\n" + "=" * 80)
print("✓ LLAMA-QUANTIZE READY")
print("=" * 80)

print(
    "Quantizer:"
)

print(
    quantizer
)

print(
    "\nNext stage:"
)

print(
    "FP16 GGUF → Q4_K_M GGUF"
)

BUILDING / LOCATING LLAMA-QUANTIZE
llama.cpp source:
/kaggle/working/llama.cpp

Build directory:
/kaggle/working/llama.cpp/build

Binary directory:
/kaggle/working/llama.cpp/build/bin

VALIDATING LLAMA.CPP SOURCE
✓ CMakeLists.txt found:
/kaggle/working/llama.cpp/CMakeLists.txt

SEARCHING FOR EXISTING LLAMA-QUANTIZE

No existing llama-quantize found.

CONFIGURING CMAKE
Running:
cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DGGML_NATIVE=OFF -DGGML_CUDA=OFF -DLLAMA_BUILD_TOOLS=ON

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile fea

In [33]:
# %% [code]
# =============================================================================
# CELL 38 — QUANTIZE TO Q4_K_M
# =============================================================================

Q4_GGUF = GGUF_DIR / "gemma3-financial-intelligence-Q4_K_M.gguf"

cmd = [
    str(quantizer),
    str(HF_GGUF),
    str(Q4_GGUF),
    "Q4_K_M",
]

print("Running:")
print(" ".join(map(str, cmd)))

subprocess.run(cmd, check=True)

print("✓ Quantized GGUF created:")
print(Q4_GGUF)

if Q4_GGUF.exists():
    print("Size GB:", round(Q4_GGUF.stat().st_size / 2**30, 3))

Running:
/kaggle/working/llama.cpp/build/bin/llama-quantize /kaggle/working/models/gguf/gemma3-financial-intelligence-f16.gguf /kaggle/working/models/gguf/gemma3-financial-intelligence-Q4_K_M.gguf Q4_K_M


version: 0.1.0-dev (build 1, commit fa88ae9)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/models/gguf/gemma3-financial-intelligence-f16.gguf' to '/kaggle/working/models/gguf/gemma3-financial-intelligence-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 34 key-value pairs and 236 tensors from /kaggle/working/models/gguf/gemma3-financial-intelligence-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                               general.


llama_quantize: quantize time =  4881.34 ms
llama_quantize:    total time =  4881.34 ms
✓ Quantized GGUF created:
/kaggle/working/models/gguf/gemma3-financial-intelligence-Q4_K_M.gguf
Size GB: 0.236


In [34]:
# %% [code]
# =============================================================================
# CELL 39 — GGUF ARTIFACT CHECK
# =============================================================================

def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"exists": False}
    return {
        "exists": True,
        "path": str(path),
        "size_mb": round(path.stat().st_size / 2**20, 2),
    }

gguf_report = {
    "merged_model": file_info(MERGED_DIR),
    "fp16_gguf": file_info(HF_GGUF),
    "q4_k_m_gguf": file_info(Q4_GGUF),
    "deployment_target": "llama.cpp-compatible GGUF",
    "quantization": "Q4_K_M",
}

with open(
    EXPORT_DIR / "gguf_export_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(gguf_report, f, indent=2)

print(json.dumps(gguf_report, indent=2))

{
  "merged_model": {
    "exists": true,
    "path": "/kaggle/working/models/merged",
    "size_mb": 0.0
  },
  "fp16_gguf": {
    "exists": true,
    "path": "/kaggle/working/models/gguf/gemma3-financial-intelligence-f16.gguf",
    "size_mb": 517.69
  },
  "q4_k_m_gguf": {
    "exists": true,
    "path": "/kaggle/working/models/gguf/gemma3-financial-intelligence-Q4_K_M.gguf",
    "size_mb": 241.39
  },
  "deployment_target": "llama.cpp-compatible GGUF",
  "quantization": "Q4_K_M"
}


In [35]:
# %% [code]
# =============================================================================
# CELL 40 — FINAL MODEL CARD / REPRODUCIBILITY REPORT
# =============================================================================

final_report = {
    "project": "ADTC 2026 Gemma 3 Financial Intelligence",
    "base_model": BASE_MODEL,
    "training": {
        "lora_rank": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "max_length": MAX_LENGTH,
        "dtype": str(MODEL_DTYPE),
    },
    "dataset": {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df),
    },
    "financial_intelligence": {
        "balance_tracking": True,
        "balance_field": "balance_after",
        "cash_flow": True,
        "spending_by_category": True,
        "spending_by_entity": True,
        "loan_affordability_heuristic": True,
    },
    "evaluation": evaluation,
    "artifacts": {
        "lora_adapter": str(LORA_DIR),
        "merged_model": str(MERGED_DIR),
        "financial_snapshot": str(EXPORT_DIR / "financial_intelligence_snapshot.json"),
        "ledger_csv": str(EXPORT_DIR / "transaction_ledger.csv"),
        "fp16_gguf": str(HF_GGUF),
        "q4_k_m_gguf": str(Q4_GGUF),
    },
    "important_note": (
        "Loan amount is an illustrative affordability heuristic, "
        "not a lender decision."
    ),
}

with open(
    REPORTS_DIR / "final_model_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False, default=str)

print("=" * 80)
print("PIPELINE COMPLETE")
print("=" * 80)

for key, value in final_report["artifacts"].items():
    print(f"{key:25s}: {value}")

print("\n✓ Financial extraction + balance tracking")
print("✓ DataFrame ledger")
print("✓ Cash-flow analytics")
print("✓ Financial question engine")
print("✓ LoRA adapter")
print("✓ Merged model")
print("✓ FP16 GGUF")
print("✓ Q4_K_M GGUF")

PIPELINE COMPLETE
lora_adapter             : /kaggle/working/models/lora
merged_model             : /kaggle/working/models/merged
financial_snapshot       : /kaggle/working/export/financial_intelligence_snapshot.json
ledger_csv               : /kaggle/working/export/transaction_ledger.csv
fp16_gguf                : /kaggle/working/models/gguf/gemma3-financial-intelligence-f16.gguf
q4_k_m_gguf              : /kaggle/working/models/gguf/gemma3-financial-intelligence-Q4_K_M.gguf

✓ Financial extraction + balance tracking
✓ DataFrame ledger
✓ Cash-flow analytics
✓ Financial question engine
✓ LoRA adapter
✓ Merged model
✓ FP16 GGUF
✓ Q4_K_M GGUF
